# 04 — Entraînement et évaluation des modèles de régression

## Objectif

Ce notebook est consacré à l'entraînement, à la comparaison et à l'évaluation
de plusieurs modèles de régression destinés à prédire les émissions de CO₂ WLTP.

La variable cible est :

`co2_wltp_g_km`

Les données utilisées proviennent directement du preprocessing validé dans le
notebook précédent :

`03_train_test_preprocessing.ipynb`

Les jeux suivants sont chargés depuis `data/processed/` :

- `X_train_processed.parquet` ;
- `X_test_processed.parquet` ;
- `y_train.parquet` ;
- `y_test.parquet`.

La méthodologie repose sur les principes suivants :

- les modèles sont entraînés exclusivement sur les données d'entraînement ;
- le jeu de test reste réservé à l'évaluation finale ;
- plusieurs familles de modèles de régression seront comparées ;
- les performances seront évaluées avec plusieurs métriques complémentaires ;
- les résultats Train et Test seront comparés afin d'identifier un éventuel
  surapprentissage ;
- le modèle retenu devra présenter un bon compromis entre performance,
  généralisation et coût de calcul.

Les principales métriques utilisées seront :

- **MAE** — Mean Absolute Error ;
- **RMSE** — Root Mean Squared Error ;
- **R²** — coefficient de détermination.

Les modèles seront étudiés progressivement, en commençant par une baseline
linéaire avant d'évaluer des modèles non linéaires plus complexes.

## 1. Chargement des données prétraitées

### Objectif

Cette étape charge les jeux Train et Test produits par le pipeline de
preprocessing.

Aucune nouvelle transformation des variables n'est réalisée dans ce notebook.

Les données chargées doivent conserver exactement la structure validée à la fin
du notebook `03_train_test_preprocessing.ipynb`.

Une première vérification porte sur :

- l'existence des fichiers ;
- les dimensions de Train et Test ;
- la cohérence entre les variables explicatives et leurs cibles ;
- l'identité de la structure des variables entre Train et Test.

In [1]:
# ---------------------------------------------------------------------
# 1 - Chargement des données prétraitées
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------
# 1. Détermination de la racine du projet
# ---------------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.parent != project_root
    and not (project_root / "pyproject.toml").exists()
):
    project_root = project_root.parent

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Impossible de déterminer la racine du projet."
    )


# ---------------------------------------------------------------------
# 2. Répertoire des données prétraitées
# ---------------------------------------------------------------------

processed_data_dir = (
    project_root
    / "data"
    / "processed"
)


# ---------------------------------------------------------------------
# 3. Chemins des fichiers
# ---------------------------------------------------------------------

x_train_path = (
    processed_data_dir
    / "X_train_processed.parquet"
)

x_test_path = (
    processed_data_dir
    / "X_test_processed.parquet"
)

y_train_path = (
    processed_data_dir
    / "y_train.parquet"
)

y_test_path = (
    processed_data_dir
    / "y_test.parquet"
)


# ---------------------------------------------------------------------
# 4. Vérification de l'existence des fichiers
# ---------------------------------------------------------------------

required_files = {
    "X_train_processed": x_train_path,
    "X_test_processed": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
}

missing_files = [
    name
    for name, path in required_files.items()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Fichiers prétraités manquants : "
        + ", ".join(missing_files)
    )


# ---------------------------------------------------------------------
# 5. Chargement
# ---------------------------------------------------------------------

X_train = pd.read_parquet(
    x_train_path
)

X_test = pd.read_parquet(
    x_test_path
)

y_train = (
    pd.read_parquet(y_train_path)
    .squeeze("columns")
)

y_test = (
    pd.read_parquet(y_test_path)
    .squeeze("columns")
)


# ---------------------------------------------------------------------
# 6. Vérifications de cohérence
# ---------------------------------------------------------------------

if len(X_train) != len(y_train):
    raise ValueError(
        "Incohérence entre X_train et y_train."
    )

if len(X_test) != len(y_test):
    raise ValueError(
        "Incohérence entre X_test et y_test."
    )

if (
    X_train.columns.tolist()
    != X_test.columns.tolist()
):
    raise ValueError(
        "X_train et X_test ne possèdent pas "
        "la même structure de variables."
    )


# ---------------------------------------------------------------------
# 7. Rapport
# ---------------------------------------------------------------------

datasets_report_df = pd.DataFrame(
    [
        {
            "dataset": "X_train",
            "observations": X_train.shape[0],
            "variables": X_train.shape[1],
        },
        {
            "dataset": "X_test",
            "observations": X_test.shape[0],
            "variables": X_test.shape[1],
        },
        {
            "dataset": "y_train",
            "observations": len(y_train),
            "variables": 1,
        },
        {
            "dataset": "y_test",
            "observations": len(y_test),
            "variables": 1,
        },
    ]
)

display(
    datasets_report_df
)

print(
    "\n✅ Données prétraitées chargées et cohérentes."
)

,dataset,observations,variables
0,X_train,8606547,41
1,X_test,2151637,41
2,y_train,8606547,1
3,y_test,2151637,1



✅ Données prétraitées chargées et cohérentes.


## 2. Définition de la méthodologie d'évaluation

### 2.1 Objectif

L'objectif de cette étape est de définir un protocole d'évaluation commun à
l'ensemble des modèles de régression qui seront entraînés dans ce notebook.

La variable cible est :

`co2_wltp_g_km`

Elle représente les émissions de CO₂ WLTP exprimées en g/km.

Les modèles seront entraînés sur `X_train` et `y_train`, puis évalués sur
`X_test` et `y_test`.


### 2.2 Métriques retenues

Trois métriques complémentaires sont utilisées pour comparer les performances
des modèles.


#### MAE — Mean Absolute Error

La MAE mesure la moyenne des écarts absolus entre les valeurs réelles et les
valeurs prédites.

Elle s'exprime dans la même unité que la variable cible, ici en **g CO₂/km**.

Une MAE faible indique que les prédictions sont, en moyenne, proches des
valeurs réelles.


#### RMSE — Root Mean Squared Error

La RMSE mesure également l'écart entre les valeurs réelles et prédites, mais
accorde davantage de poids aux erreurs importantes en raison de l'élévation
au carré des écarts.

Elle s'exprime également en **g CO₂/km**.

Une RMSE sensiblement supérieure à la MAE peut notamment signaler la présence
de certaines erreurs de prédiction importantes.


#### R² — Coefficient de détermination

Le coefficient R² mesure la proportion de la variabilité de la cible expliquée
par le modèle.

Plus R² est proche de `1`, plus le modèle explique correctement la variabilité
des émissions de CO₂ observées.

Un R² élevé ne doit cependant pas être interprété isolément. Il sera analysé
conjointement avec la MAE, la RMSE et l'écart de performance entre les données
d'entraînement et de test.


### 2.3 Évaluation sur Train et Test

Pour chaque modèle, les métriques seront calculées sur :

- le jeu d'entraînement (`Train`) ;
- le jeu de test (`Test`).

Cette double évaluation permettra notamment d'analyser la capacité de
généralisation du modèle.

Un modèle très performant sur Train mais sensiblement moins performant sur Test
peut présenter un phénomène de surapprentissage.


### 2.4 Temps d'entraînement

Le temps nécessaire à l'entraînement sera également mesuré.

Cette information est particulièrement importante dans ce projet puisque le
jeu d'entraînement complet contient plusieurs millions d'observations.

La sélection finale ne reposera donc pas uniquement sur la précision
prédictive, mais également sur le compromis entre :

- performance ;
- généralisation ;
- complexité ;
- coût de calcul.


### 2.5 Principe de comparaison

Tous les modèles seront évalués selon le même protocole afin de permettre une
comparaison homogène.

Les résultats seront progressivement enregistrés dans un tableau comparatif
contenant notamment :

| Modèle | MAE Train | MAE Test | RMSE Train | RMSE Test | R² Train | R² Test | Temps d'entraînement |
|---|---:|---:|---:|---:|---:|---:|---:|

Ce tableau servira de support à la sélection du modèle candidat pour la suite
du pipeline MLOps.

In [2]:
# ---------------------------------------------------------------------
# 2.6 - Fonction commune d'évaluation des modèles de régression
# ---------------------------------------------------------------------

import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ---------------------------------------------------------------------
# Tableau qui recevra progressivement les résultats des modèles
# ---------------------------------------------------------------------

regression_results = []


# ---------------------------------------------------------------------
# Fonction d'évaluation
# ---------------------------------------------------------------------

def evaluate_regression_model(
    model_name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
):
    """
    Entraîne un modèle de régression et calcule ses performances
    sur les jeux Train et Test.

    Parameters
    ----------
    model_name : str
        Nom du modèle.

    model :
        Estimateur compatible avec l'API scikit-learn.

    X_train, y_train :
        Données utilisées pour l'apprentissage.

    X_test, y_test :
        Données réservées à l'évaluation.

    Returns
    -------
    dict
        Métriques Train/Test et durée d'entraînement.
    """

    # -------------------------------------------------------------
    # Entraînement et mesure du temps
    # -------------------------------------------------------------

    start_time = time.perf_counter()

    model.fit(
        X_train,
        y_train,
    )

    training_time_seconds = (
        time.perf_counter()
        - start_time
    )


    # -------------------------------------------------------------
    # Prédictions
    # -------------------------------------------------------------

    y_train_pred = model.predict(
        X_train
    )

    y_test_pred = model.predict(
        X_test
    )


    # -------------------------------------------------------------
    # Métriques Train
    # -------------------------------------------------------------

    mae_train = mean_absolute_error(
        y_train,
        y_train_pred,
    )

    rmse_train = np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred,
        )
    )

    r2_train = r2_score(
        y_train,
        y_train_pred,
    )


    # -------------------------------------------------------------
    # Métriques Test
    # -------------------------------------------------------------

    mae_test = mean_absolute_error(
        y_test,
        y_test_pred,
    )

    rmse_test = np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred,
        )
    )

    r2_test = r2_score(
        y_test,
        y_test_pred,
    )


    # -------------------------------------------------------------
    # Résultats
    # -------------------------------------------------------------

    result = {
        "model": model_name,
        "mae_train": mae_train,
        "mae_test": mae_test,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "r2_train": r2_train,
        "r2_test": r2_test,
        "training_time_seconds": training_time_seconds,
    }

    regression_results.append(
        result
    )

    return result

## 3. Stratégie d'entraînement TEST / FULL

### Objectif

Le jeu d'entraînement complet contient plusieurs millions d'observations.

Afin d'éviter de lancer systématiquement des entraînements coûteux pendant
la phase de développement et de comparaison des modèles, deux modes
d'utilisation sont distingués :

- **mode TEST** : entraînement sur un sous-échantillon de `X_train` et `y_train`
  afin de valider rapidement le fonctionnement du modèle, le code et les
  métriques ;
- **mode FULL** : entraînement sur l'intégralité de `X_train` et `y_train`
  une fois la méthodologie validée.

Le jeu `X_test` / `y_test` reste réservé à l'évaluation finale et n'est pas
utilisé pour l'apprentissage.

Cette organisation permet de séparer :

- la phase de développement rapide ;
- la phase d'entraînement complète ;
- l'évaluation finale sur des données non utilisées pour l'apprentissage.


### Sous-échantillon de développement

Le sous-échantillon TEST est extrait exclusivement à partir du jeu
d'entraînement.

Il ne modifie donc pas le jeu de test final.

Le nombre d'observations utilisé en mode TEST est défini explicitement et peut
être ajusté selon les ressources disponibles et la complexité du modèle.

L'échantillonnage est reproductible grâce à l'utilisation d'un
`random_state`.

In [3]:
# ---------------------------------------------------------------------
# 3 - Préparation d'un sous-échantillon d'entraînement pour le mode TEST
# ---------------------------------------------------------------------

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

TRAIN_SAMPLE_SIZE = 500_000
RANDOM_STATE = 42


# ---------------------------------------------------------------------
# 2. Vérification de la taille disponible
# ---------------------------------------------------------------------

if TRAIN_SAMPLE_SIZE > len(X_train):
    raise ValueError(
        "TRAIN_SAMPLE_SIZE est supérieur au nombre "
        "d'observations disponibles dans X_train."
    )


# ---------------------------------------------------------------------
# 3. Sous-échantillonnage reproductible de Train
# ---------------------------------------------------------------------

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------------------
# 4. Rapport
# ---------------------------------------------------------------------

training_sample_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train FULL",
            "observations": len(X_train),
            "variables": X_train.shape[1],
        },
        {
            "dataset": "Train TEST",
            "observations": len(X_train_sample),
            "variables": X_train_sample.shape[1],
        },
        {
            "dataset": "Test final",
            "observations": len(X_test),
            "variables": X_test.shape[1],
        },
    ]
)

display(training_sample_report_df)

print(
    "\n✅ Sous-échantillon d'entraînement créé "
    "sans modifier le jeu de test final."
)

,dataset,observations,variables
0,Train FULL,8606547,41
1,Train TEST,500000,41
2,Test final,2151637,41



✅ Sous-échantillon d'entraînement créé sans modifier le jeu de test final.


## 4. Modèle baseline — Régression Ridge

### Objectif

Le premier modèle utilisé comme référence est une **Régression Ridge**.

La Régression Ridge est un modèle linéaire régularisé. Elle permet d'établir
une baseline simple avant d'évaluer des modèles non linéaires plus complexes.

Cette première expérimentation est réalisée sur le sous-échantillon
d'entraînement `Train TEST`.

Le modèle est entraîné à partir de :

- `X_train_sample` ;
- `y_train_sample`.

Il est ensuite évalué :

- sur ce même sous-échantillon d'entraînement afin de mesurer sa capacité
  d'ajustement ;
- sur `X_test` / `y_test`, qui restent réservés à l'évaluation finale.

Les métriques utilisées sont :

- MAE ;
- RMSE ;
- R².

Le temps d'entraînement est également mesuré.

Cette baseline servira ensuite de point de comparaison avec les modèles
non linéaires.

In [4]:
# ---------------------------------------------------------------------
# 4 - Modèle baseline : Régression Ridge
# ---------------------------------------------------------------------

from sklearn.linear_model import Ridge


# ---------------------------------------------------------------------
# 1. Définition du modèle
# ---------------------------------------------------------------------

ridge_model = Ridge(
    alpha=1.0
)


# ---------------------------------------------------------------------
# 2. Entraînement et évaluation
# ---------------------------------------------------------------------

ridge_result = evaluate_regression_model(
    model_name="Ridge",
    model=ridge_model,
    X_train=X_train_sample,
    y_train=y_train_sample,
    X_test=X_test,
    y_test=y_test,
)


# ---------------------------------------------------------------------
# 3. Rapport des performances
# ---------------------------------------------------------------------

ridge_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train TEST",
            "MAE_g_co2_km": ridge_result["mae_train"],
            "RMSE_g_co2_km": ridge_result["rmse_train"],
            "R2": ridge_result["r2_train"],
        },
        {
            "dataset": "Test final",
            "MAE_g_co2_km": ridge_result["mae_test"],
            "RMSE_g_co2_km": ridge_result["rmse_test"],
            "R2": ridge_result["r2_test"],
        },
    ]
)

display(
    ridge_report_df
)


print(
    f"\nTemps d'entraînement Ridge : "
    f"{ridge_result['training_time_seconds']:.2f} secondes"
)

,dataset,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train TEST,6.698968,11.335787,0.961771
1,Test final,6.704354,11.362967,0.961458



Temps d'entraînement Ridge : 0.42 secondes


## 5. Random Forest Regressor

### 5.1 Objectif

Après l'établissement d'une baseline linéaire avec la Régression Ridge,
un modèle non linéaire de type **Random Forest Regressor** est évalué.

Random Forest est un modèle d'ensemble basé sur plusieurs arbres de décision.

Contrairement à Ridge, il peut modéliser des relations non linéaires et des
interactions complexes entre les caractéristiques des véhicules sans imposer
une relation linéaire entre les variables explicatives et les émissions de CO₂.

Cette première expérimentation est réalisée sur le sous-échantillon
`Train TEST` de 500 000 observations.

Le modèle est ensuite évalué :

- sur `Train TEST`, afin de mesurer son niveau d'ajustement aux données
  d'apprentissage ;
- sur le `Test final`, afin d'évaluer sa capacité de généralisation.

Les mêmes métriques que pour Ridge sont utilisées :

- MAE ;
- RMSE ;
- R² ;
- temps d'entraînement.

L'écart entre les performances Train et Test sera particulièrement surveillé,
car les modèles basés sur des arbres peuvent présenter un surapprentissage
plus important qu'un modèle linéaire régularisé.

À ce stade, l'objectif n'est pas encore d'effectuer une optimisation exhaustive
des hyperparamètres, mais d'obtenir une première mesure de performance du
Random Forest dans des conditions de calcul maîtrisées.

In [5]:
# ---------------------------------------------------------------------
# 5 - Random Forest Regressor
# ---------------------------------------------------------------------

from sklearn.ensemble import RandomForestRegressor


# ---------------------------------------------------------------------
# 1. Définition du modèle
# ---------------------------------------------------------------------

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


# ---------------------------------------------------------------------
# 2. Entraînement et évaluation
# ---------------------------------------------------------------------

random_forest_result = evaluate_regression_model(
    model_name="Random Forest",
    model=random_forest_model,
    X_train=X_train_sample,
    y_train=y_train_sample,
    X_test=X_test,
    y_test=y_test,
)


# ---------------------------------------------------------------------
# 3. Rapport des performances
# ---------------------------------------------------------------------

random_forest_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train TEST",
            "MAE_g_co2_km": random_forest_result["mae_train"],
            "RMSE_g_co2_km": random_forest_result["rmse_train"],
            "R2": random_forest_result["r2_train"],
        },
        {
            "dataset": "Test final",
            "MAE_g_co2_km": random_forest_result["mae_test"],
            "RMSE_g_co2_km": random_forest_result["rmse_test"],
            "R2": random_forest_result["r2_test"],
        },
    ]
)

display(
    random_forest_report_df
)


print(
    f"\nTemps d'entraînement Random Forest : "
    f"{random_forest_result['training_time_seconds']:.2f} secondes"
)

,dataset,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train TEST,0.378939,1.594698,0.999243
1,Test final,0.511380,3.050825,0.997222



Temps d'entraînement Random Forest : 157.08 secondes


## 6. HistGradientBoostingRegressor

### 6.1 Objectif

Après la Régression Ridge et le Random Forest Regressor, une troisième famille
de modèles est évaluée : le **Gradient Boosting**.

Le modèle retenu pour cette expérimentation est
`HistGradientBoostingRegressor`.

Contrairement au Random Forest, qui construit plusieurs arbres de manière
indépendante, le Gradient Boosting construit progressivement une succession
d'arbres.

Chaque nouvelle étape cherche à améliorer les erreurs produites par les étapes
précédentes.

### Pourquoi HistGradientBoostingRegressor ?

Le dataset complet du projet contient plusieurs millions d'observations.

`HistGradientBoostingRegressor` utilise une représentation des variables par
intervalles (histogrammes), ce qui le rend particulièrement adapté aux jeux de
données volumineux.

Cette première expérimentation est réalisée sur le sous-échantillon
`Train TEST` de 500 000 observations.

Le modèle sera évalué avec le même protocole que les modèles précédents :

- MAE sur Train et Test ;
- RMSE sur Train et Test ;
- R² sur Train et Test ;
- temps d'entraînement.

Les résultats seront ensuite comparés à ceux de Ridge et Random Forest.

À ce stade, aucune optimisation exhaustive des hyperparamètres n'est réalisée.
L'objectif est d'obtenir une première performance de référence pour cette
famille de modèles.

In [6]:
# ---------------------------------------------------------------------
# 6.1 - HistGradientBoostingRegressor
# ---------------------------------------------------------------------

from sklearn.ensemble import HistGradientBoostingRegressor


# ---------------------------------------------------------------------
# 1. Définition du modèle
# ---------------------------------------------------------------------

hist_gradient_boosting_model = HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------------------
# 2. Entraînement et évaluation
# ---------------------------------------------------------------------

hist_gradient_boosting_result = evaluate_regression_model(
    model_name="HistGradientBoosting",
    model=hist_gradient_boosting_model,
    X_train=X_train_sample,
    y_train=y_train_sample,
    X_test=X_test,
    y_test=y_test,
)


# ---------------------------------------------------------------------
# 3. Rapport des performances
# ---------------------------------------------------------------------

hist_gradient_boosting_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train TEST",
            "MAE_g_co2_km": hist_gradient_boosting_result["mae_train"],
            "RMSE_g_co2_km": hist_gradient_boosting_result["rmse_train"],
            "R2": hist_gradient_boosting_result["r2_train"],
        },
        {
            "dataset": "Test final",
            "MAE_g_co2_km": hist_gradient_boosting_result["mae_test"],
            "RMSE_g_co2_km": hist_gradient_boosting_result["rmse_test"],
            "R2": hist_gradient_boosting_result["r2_test"],
        },
    ]
)

display(
    hist_gradient_boosting_report_df
)


print(
    f"\nTemps d'entraînement HistGradientBoosting : "
    f"{hist_gradient_boosting_result['training_time_seconds']:.2f} secondes"
)

,dataset,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Train TEST,1.757041,4.227916,0.994682
1,Test final,1.791337,4.588191,0.993716



Temps d'entraînement HistGradientBoosting : 13.40 secondes


### 7.2 Interprétation des résultats

La comparaison des modèles doit être réalisée selon trois axes complémentaires :

- **précision prédictive** :
  - MAE ;
  - RMSE ;
  - R² ;

- **capacité de généralisation** :
  - comparaison des performances obtenues sur Train et Test ;
  - analyse des écarts entre les métriques Train et Test ;

- **coût de calcul** :
  - temps d'entraînement du modèle.

Un modèle présentant les meilleures performances sur Train n'est pas
nécessairement le meilleur candidat si ses performances se dégradent
sensiblement sur Test.

De même, un modèle légèrement moins précis peut rester particulièrement
intéressant s'il présente une meilleure stabilité et un coût d'entraînement
nettement inférieur.

La sélection du ou des modèles candidats doit donc reposer sur un compromis
entre :

- précision ;
- généralisation ;
- robustesse ;
- temps de calcul.

Les résultats numériques de cette comparaison sont générés dynamiquement à
partir des métriques obtenues lors de l'exécution courante du notebook.

In [7]:
# ---------------------------------------------------------------------
# 7.1 - Comparaison consolidée des modèles de régression
# ---------------------------------------------------------------------

regression_comparison_df = (
    pd.DataFrame(regression_results)
    .copy()
)


# ---------------------------------------------------------------------
# 1. Calcul des écarts Train / Test
# ---------------------------------------------------------------------

regression_comparison_df["mae_gap"] = (
    regression_comparison_df["mae_test"]
    - regression_comparison_df["mae_train"]
)

regression_comparison_df["rmse_gap"] = (
    regression_comparison_df["rmse_test"]
    - regression_comparison_df["rmse_train"]
)

regression_comparison_df["r2_gap"] = (
    regression_comparison_df["r2_train"]
    - regression_comparison_df["r2_test"]
)


# ---------------------------------------------------------------------
# 2. Organisation des colonnes
# ---------------------------------------------------------------------

regression_comparison_df = regression_comparison_df[
    [
        "model",
        "mae_train",
        "mae_test",
        "mae_gap",
        "rmse_train",
        "rmse_test",
        "rmse_gap",
        "r2_train",
        "r2_test",
        "r2_gap",
        "training_time_seconds",
    ]
]


# ---------------------------------------------------------------------
# 3. Classement selon la MAE Test
# ---------------------------------------------------------------------

regression_comparison_df = (
    regression_comparison_df
    .sort_values(
        by="mae_test",
        ascending=True,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 4. Version destinée uniquement à l'affichage
# ---------------------------------------------------------------------

regression_comparison_display_df = (
    regression_comparison_df.copy()
)

numeric_columns = (
    regression_comparison_display_df
    .select_dtypes(include="number")
    .columns
)

regression_comparison_display_df[
    numeric_columns
] = (
    regression_comparison_display_df[
        numeric_columns
    ]
    .round(4)
)


display(regression_comparison_display_df)

,model,mae_train,mae_test,mae_gap,rmse_train,rmse_test,rmse_gap,r2_train,r2_test,r2_gap,training_time_seconds
0,Random Forest,0.3789,0.5114,0.1324,1.5947,3.0508,1.4561,0.9992,0.9972,0.0020,157.0765
1,HistGradientBoosting,1.7570,1.7913,0.0343,4.2279,4.5882,0.3603,0.9947,0.9937,0.0010,13.3964
2,Ridge,6.6990,6.7044,0.0054,11.3358,11.3630,0.0272,0.9618,0.9615,0.0003,0.4220


### 7.2 Analyse dynamique des performances

L'interprétation des résultats est réalisée automatiquement à partir du tableau
comparatif généré lors de l'exécution courante.

L'analyse identifie notamment :

- le modèle présentant la plus faible MAE sur le jeu de test ;
- le modèle présentant la plus faible RMSE sur le jeu de test ;
- le modèle présentant le meilleur R² sur le jeu de test ;
- le modèle ayant le temps d'entraînement le plus faible ;
- les écarts de performance entre Train et Test.

Cette approche évite d'inscrire des résultats numériques en dur dans le
notebook et permet de conserver la même analyse lors d'une exécution ultérieure
avec d'autres données ou dans un contexte d'entraînement FULL.

In [8]:
# ---------------------------------------------------------------------
# 7.2 - Analyse dynamique des performances
# ---------------------------------------------------------------------

# Meilleurs modèles selon les différentes dimensions
best_mae_model = regression_comparison_df.loc[
    regression_comparison_df["mae_test"].idxmin()
]

best_rmse_model = regression_comparison_df.loc[
    regression_comparison_df["rmse_test"].idxmin()
]

best_r2_model = regression_comparison_df.loc[
    regression_comparison_df["r2_test"].idxmax()
]

fastest_model = regression_comparison_df.loc[
    regression_comparison_df["training_time_seconds"].idxmin()
]


# ---------------------------------------------------------------------
# Rapport synthétique
# ---------------------------------------------------------------------

print("COMPARAISON DES MODÈLES DE RÉGRESSION")
print("=" * 60)

print("\n1. Meilleure MAE sur le Test final")
print(
    f"   {best_mae_model['model']} : "
    f"{best_mae_model['mae_test']:.4f} g CO₂/km"
)

print("\n2. Meilleure RMSE sur le Test final")
print(
    f"   {best_rmse_model['model']} : "
    f"{best_rmse_model['rmse_test']:.4f} g CO₂/km"
)

print("\n3. Meilleur R² sur le Test final")
print(
    f"   {best_r2_model['model']} : "
    f"{best_r2_model['r2_test']:.4f}"
)

print("\n4. Temps d'entraînement le plus faible")
print(
    f"   {fastest_model['model']} : "
    f"{fastest_model['training_time_seconds']:.2f} secondes"
)


# ---------------------------------------------------------------------
# Analyse Train / Test de chaque modèle
# ---------------------------------------------------------------------

print("\n5. Écarts de généralisation")
print("-" * 60)

for _, row in regression_comparison_df.iterrows():

    print(f"\n   {row['model']}")

    print(
        f"   - Écart MAE  : "
        f"{row['mae_gap']:.4f} g CO₂/km"
    )

    print(
        f"   - Écart RMSE : "
        f"{row['rmse_gap']:.4f} g CO₂/km"
    )

    print(
        f"   - Écart R²   : "
        f"{row['r2_gap']:.4f}"
    )

COMPARAISON DES MODÈLES DE RÉGRESSION

1. Meilleure MAE sur le Test final
   Random Forest : 0.5114 g CO₂/km

2. Meilleure RMSE sur le Test final
   Random Forest : 3.0508 g CO₂/km

3. Meilleur R² sur le Test final
   Random Forest : 0.9972

4. Temps d'entraînement le plus faible
   Ridge : 0.42 secondes

5. Écarts de généralisation
------------------------------------------------------------

   Random Forest
   - Écart MAE  : 0.1324 g CO₂/km
   - Écart RMSE : 1.4561 g CO₂/km
   - Écart R²   : 0.0020

   HistGradientBoosting
   - Écart MAE  : 0.0343 g CO₂/km
   - Écart RMSE : 0.3603 g CO₂/km
   - Écart R²   : 0.0010

   Ridge
   - Écart MAE  : 0.0054 g CO₂/km
   - Écart RMSE : 0.0272 g CO₂/km
   - Écart R²   : 0.0003


### 7.3 Conclusion de la comparaison et sélection des modèles candidats

La comparaison met en évidence des profils différents entre les trois modèles
évalués.

#### Ridge

Ridge présente les écarts Train/Test les plus faibles ainsi que le temps
d'entraînement le plus court.

Cette stabilité doit cependant être interprétée conjointement avec ses
performances prédictives, qui restent inférieures à celles des deux modèles
non linéaires.

Ridge conserve donc son rôle de **baseline de référence**, mais n'est pas
retenu comme candidat prioritaire pour la phase d'optimisation.


#### Random Forest Regressor

Random Forest obtient les meilleures performances prédictives sur le jeu de
test pour les trois métriques étudiées :

- MAE ;
- RMSE ;
- R².

Il présente toutefois des écarts Train/Test plus importants que les deux
autres modèles ainsi qu'un coût d'entraînement sensiblement supérieur.

Random Forest est donc retenu comme **candidat orienté vers la maximisation
de la performance prédictive**.


#### HistGradientBoostingRegressor

HistGradientBoosting présente des performances prédictives légèrement
inférieures à celles de Random Forest, mais conserve un niveau de performance
élevé sur le jeu de test.

Il présente également :

- des écarts Train/Test plus faibles que Random Forest ;
- un temps d'entraînement nettement inférieur à Random Forest.

HistGradientBoosting est donc retenu comme **candidat orienté vers le compromis
entre performance prédictive, généralisation et efficacité computationnelle**.


### Modèles retenus pour la suite

Deux modèles sont retenus pour la phase suivante :

1. **Random Forest Regressor**
   - candidat privilégiant la performance prédictive ;

2. **HistGradientBoostingRegressor**
   - candidat privilégiant le compromis entre performance, généralisation
     et coût de calcul.

Aucun modèle final n'est sélectionné à ce stade.

La prochaine étape consiste à optimiser les hyperparamètres de ces deux
modèles selon une stratégie compatible avec le volume important du dataset.

## 8. Optimisation des hyperparamètres

### Objectif

Les configurations utilisées jusqu'à présent constituent des configurations
initiales et non des configurations optimisées.

L'objectif est maintenant de rechercher de meilleurs hyperparamètres pour les
deux modèles candidats :

- Random Forest Regressor ;
- HistGradientBoostingRegressor.

Compte tenu du volume du dataset, une recherche exhaustive de type
`GridSearchCV` sur l'ensemble des données serait particulièrement coûteuse.

L'optimisation sera donc réalisée selon une stratégie progressive permettant
de maîtriser le temps de calcul tout en conservant une méthodologie
reproductible.

### 8.1 Constitution des données pour l'optimisation des hyperparamètres

### Objectif

L'optimisation des hyperparamètres ne doit pas utiliser le jeu `Test final`.

Le jeu de test doit rester indépendant afin de fournir, à la fin du processus,
une évaluation aussi objective que possible du modèle sélectionné.

La recherche des hyperparamètres est donc réalisée uniquement à partir des
données d'entraînement.

À partir du sous-échantillon `Train TEST`, deux ensembles sont constitués :

- un ensemble d'apprentissage pour entraîner les différentes configurations ;
- un ensemble de validation pour comparer ces configurations.

Cette séparation permet de sélectionner les hyperparamètres sans utiliser
`X_test` ni `y_test`.

Le découpage est réalisé de manière reproductible à l'aide d'un
`random_state`.

Le jeu `Test final` reste inchangé et sera utilisé uniquement après la sélection
des meilleures configurations.

In [9]:
# ---------------------------------------------------------------------
# 8.1 - Constitution des données pour l'optimisation
# ---------------------------------------------------------------------

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

VALIDATION_SIZE = 0.20


# ---------------------------------------------------------------------
# 2. Découpage du sous-échantillon Train TEST
#
# Source :
# X_train_sample / y_train_sample = 500 000 observations extraites
# du jeu d'entraînement.
#
# Le Test final X_test / y_test n'est PAS utilisé pour le tuning.
# ---------------------------------------------------------------------

(
    X_tuning_train,
    X_tuning_validation,
    y_tuning_train,
    y_tuning_validation,
) = train_test_split(
    X_train_sample,
    y_train_sample,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------------------
# 3. Vérification de la cohérence X / y
# ---------------------------------------------------------------------

if len(X_tuning_train) != len(y_tuning_train):
    raise ValueError(
        "Incohérence entre X_tuning_train et y_tuning_train."
    )

if len(X_tuning_validation) != len(y_tuning_validation):
    raise ValueError(
        "Incohérence entre X_tuning_validation "
        "et y_tuning_validation."
    )


# ---------------------------------------------------------------------
# 4. Vérification de la structure des variables
# ---------------------------------------------------------------------

if (
    X_tuning_train.columns.tolist()
    != X_tuning_validation.columns.tolist()
):
    raise ValueError(
        "Les colonnes du jeu d'apprentissage et du jeu "
        "de validation sont différentes."
    )

if (
    X_tuning_train.columns.tolist()
    != X_test.columns.tolist()
):
    raise ValueError(
        "Les colonnes utilisées pour le tuning et celles "
        "du Test final sont différentes."
    )


# ---------------------------------------------------------------------
# 5. Vérification du découpage
# ---------------------------------------------------------------------

if (
    len(X_tuning_train)
    + len(X_tuning_validation)
    != len(X_train_sample)
):
    raise ValueError(
        "Le découpage du Train TEST est incohérent."
    )


# ---------------------------------------------------------------------
# 6. Tableau explicatif
# ---------------------------------------------------------------------

tuning_split_report_df = pd.DataFrame(
    [
        {
            "jeu_de_donnees": "Train TEST initial",
            "source": "X_train_sample",
            "observations": len(X_train_sample),
            "variables": X_train_sample.shape[1],
            "utilisation": "Source du tuning",
        },
        {
            "jeu_de_donnees": "Apprentissage tuning",
            "source": "Train TEST initial",
            "observations": len(X_tuning_train),
            "variables": X_tuning_train.shape[1],
            "utilisation": (
                "Entraînement des configurations "
                "d'hyperparamètres"
            ),
        },
        {
            "jeu_de_donnees": "Validation tuning",
            "source": "Train TEST initial",
            "observations": len(X_tuning_validation),
            "variables": X_tuning_validation.shape[1],
            "utilisation": (
                "Comparaison et sélection "
                "des hyperparamètres"
            ),
        },
        {
            "jeu_de_donnees": "Test final",
            "source": "X_test_processed.parquet",
            "observations": len(X_test),
            "variables": X_test.shape[1],
            "utilisation": (
                "Évaluation finale uniquement — "
                "exclu du tuning"
            ),
        },
    ]
)

display(tuning_split_report_df)


# ---------------------------------------------------------------------
# 7. Rapport synthétique
# ---------------------------------------------------------------------

print("\nRÉPARTITION DES DONNÉES POUR L'OPTIMISATION")
print("=" * 70)

print(
    f"\nTrain TEST initial       : "
    f"{len(X_train_sample):,} observations"
)

print(
    f"├── Apprentissage tuning : "
    f"{len(X_tuning_train):,} observations "
    f"({(1 - VALIDATION_SIZE) * 100:.0f} %)"
)

print(
    f"└── Validation tuning    : "
    f"{len(X_tuning_validation):,} observations "
    f"({VALIDATION_SIZE * 100:.0f} %)"
)

print(
    f"\nTest final indépendant   : "
    f"{len(X_test):,} observations"
)

print(
    "\n✅ Le tuning utilise exclusivement les données "
    "issues de Train TEST."
)

print(
    "✅ Le Test final reste totalement exclu de "
    "l'optimisation des hyperparamètres."
)

,jeu_de_donnees,source,observations,variables,utilisation
0,Train TEST initial,X_train_sample,500000,41,Source du tuning
1,Apprentissage tuning,Train TEST initial,400000,41,Entraînement des configurations d'hyperparamètres
2,Validation tuning,Train TEST initial,100000,41,Comparaison et sélection des hyperparamètres
3,Test final,X_test_processed.parquet,2151637,41,Évaluation finale uniquement — exclu du tuning



RÉPARTITION DES DONNÉES POUR L'OPTIMISATION

Train TEST initial       : 500,000 observations
├── Apprentissage tuning : 400,000 observations (80 %)
└── Validation tuning    : 100,000 observations (20 %)

Test final indépendant   : 2,151,637 observations

✅ Le tuning utilise exclusivement les données issues de Train TEST.
✅ Le Test final reste totalement exclu de l'optimisation des hyperparamètres.


### 8.2 Définition des espaces d'hyperparamètres

### Objectif

Les configurations utilisées lors de la première comparaison des modèles
étaient des configurations initiales fixées manuellement.

L'objectif est maintenant de définir les espaces d'hyperparamètres qui seront
explorés pour les deux modèles candidats :

- Random Forest Regressor ;
- HistGradientBoostingRegressor.

Compte tenu du volume des données et du coût d'entraînement observé,
l'optimisation doit rester maîtrisée.

Une recherche exhaustive de toutes les combinaisons possibles serait
particulièrement coûteuse, notamment pour Random Forest.

Les espaces de recherche sont donc volontairement bornés autour de paramètres
ayant une influence directe sur :

- la complexité du modèle ;
- sa capacité de généralisation ;
- le risque de surapprentissage ;
- son temps d'entraînement.

La recherche des meilleures configurations sera réalisée uniquement à partir
des données dédiées au tuning définies à l'étape précédente.

Le jeu `Test final` reste exclu de cette phase.

In [10]:
# ---------------------------------------------------------------------
# 8.2 - Définition des espaces d'hyperparamètres
# ---------------------------------------------------------------------


# ---------------------------------------------------------------------
# 1. Random Forest Regressor
# ---------------------------------------------------------------------

random_forest_param_distributions = {

    # Nombre d'arbres constituant la forêt
    "n_estimators": [
        100,
        200,
        300,
    ],

    # Profondeur maximale des arbres
    "max_depth": [
        15,
        20,
        25,
        None,
    ],

    # Nombre minimum d'observations nécessaires
    # pour effectuer une séparation
    "min_samples_split": [
        2,
        5,
        10,
    ],

    # Nombre minimum d'observations dans une feuille
    "min_samples_leaf": [
        1,
        2,
        5,
    ],

    # Nombre de variables considérées à chaque séparation
    "max_features": [
        "sqrt",
        0.7,
        1.0,
    ],
}


# ---------------------------------------------------------------------
# 2. HistGradientBoostingRegressor
# ---------------------------------------------------------------------

hist_gradient_boosting_param_distributions = {

    # Vitesse d'apprentissage
    "learning_rate": [
        0.03,
        0.05,
        0.1,
    ],

    # Nombre maximal d'itérations de boosting
    "max_iter": [
        100,
        200,
        300,
    ],

    # Nombre maximal de feuilles par arbre
    "max_leaf_nodes": [
        15,
        31,
        63,
    ],

    # Nombre minimum d'observations par feuille
    "min_samples_leaf": [
        20,
        50,
        100,
    ],

    # Régularisation L2
    "l2_regularization": [
        0.0,
        0.1,
        1.0,
        5.0,
    ],
}


# ---------------------------------------------------------------------
# 3. Nombre théorique de combinaisons
# ---------------------------------------------------------------------

from math import prod


rf_combinations = prod(
    len(values)
    for values
    in random_forest_param_distributions.values()
)

hgb_combinations = prod(
    len(values)
    for values
    in hist_gradient_boosting_param_distributions.values()
)


# ---------------------------------------------------------------------
# 4. Rapport
# ---------------------------------------------------------------------

hyperparameter_spaces_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest",
            "hyperparametres": len(
                random_forest_param_distributions
            ),
            "combinaisons_theoriques": rf_combinations,
        },
        {
            "modele": "HistGradientBoosting",
            "hyperparametres": len(
                hist_gradient_boosting_param_distributions
            ),
            "combinaisons_theoriques": hgb_combinations,
        },
    ]
)

display(
    hyperparameter_spaces_df
)


print("\nESPACES D'HYPERPARAMÈTRES")
print("=" * 65)

print(
    f"\nRandom Forest : "
    f"{rf_combinations:,} combinaisons théoriques"
)

print(
    f"HistGradientBoosting : "
    f"{hgb_combinations:,} combinaisons théoriques"
)

print(
    "\n✅ Espaces d'hyperparamètres définis."
)

print(
    "✅ Aucun entraînement n'a encore été lancé."
)

print(
    "✅ Le Test final reste exclu du tuning."
)

,modele,hyperparametres,combinaisons_theoriques
0,Random Forest,5,324
1,HistGradientBoosting,5,324



ESPACES D'HYPERPARAMÈTRES

Random Forest : 324 combinaisons théoriques
HistGradientBoosting : 324 combinaisons théoriques

✅ Espaces d'hyperparamètres définis.
✅ Aucun entraînement n'a encore été lancé.
✅ Le Test final reste exclu du tuning.


### 8.3 Définition de la stratégie d'optimisation

### Objectif

Les espaces d'hyperparamètres définis précédemment contiennent chacun plusieurs
centaines de combinaisons possibles.

Une recherche exhaustive de type `GridSearchCV` serait trop coûteuse dans le
contexte de ce projet, notamment pour Random Forest.

La stratégie retenue repose donc sur une recherche aléatoire contrôlée avec
`RandomizedSearchCV`.

Cette méthode permet de tester un nombre limité de configurations tirées dans
l'espace défini, tout en conservant une procédure reproductible.

La sélection des configurations repose sur la **MAE**, utilisée comme métrique
principale de comparaison.

Comme Scikit-learn cherche à maximiser le score, la métrique utilisée sera :

`neg_mean_absolute_error`

Le score retourné sera donc négatif pendant la recherche, puis reconverti en
MAE positive pour l'interprétation.

### Validation croisée

La recherche sera réalisée uniquement sur les données dédiées au tuning.

Une validation croisée à **3 folds** est retenue afin de limiter le coût de
calcul tout en évitant de sélectionner les hyperparamètres sur un seul découpage.

Ainsi, chaque configuration testée sera entraînée trois fois.

### Nombre de configurations

Compte tenu du coût observé précédemment :

- Random Forest sera testé sur un nombre volontairement limité de
  configurations ;
- HistGradientBoosting pourra être exploré plus largement en raison de son
  temps d'entraînement nettement inférieur.

Le Test final reste totalement exclu de cette phase.

La meilleure configuration obtenue lors du tuning sera ensuite réévaluée
séparément avant toute décision d'entraînement FULL.

In [11]:
# ---------------------------------------------------------------------
# 8.3 - Définition de la stratégie de tuning
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1. Paramètres généraux
# ---------------------------------------------------------------------

CV_FOLDS = 3

SCORING = "neg_mean_absolute_error"


# ---------------------------------------------------------------------
# 2. Nombre de configurations à tester
# ---------------------------------------------------------------------

RF_N_ITER = 8

HGB_N_ITER = 20


# ---------------------------------------------------------------------
# 3. Nombre total d'entraînements
# ---------------------------------------------------------------------

rf_total_fits = (
    RF_N_ITER
    * CV_FOLDS
)

hgb_total_fits = (
    HGB_N_ITER
    * CV_FOLDS
)


# ---------------------------------------------------------------------
# 4. Rapport de stratégie
# ---------------------------------------------------------------------

tuning_strategy_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest",
            "configurations_testees": RF_N_ITER,
            "folds_cv": CV_FOLDS,
            "entrainements_totaux": rf_total_fits,
            "metrique_selection": "MAE",
        },
        {
            "modele": "HistGradientBoosting",
            "configurations_testees": HGB_N_ITER,
            "folds_cv": CV_FOLDS,
            "entrainements_totaux": hgb_total_fits,
            "metrique_selection": "MAE",
        },
    ]
)

display(
    tuning_strategy_df
)


print("\nSTRATÉGIE DE TUNING")
print("=" * 65)

print(
    f"\nRandom Forest : "
    f"{RF_N_ITER} configurations × "
    f"{CV_FOLDS} folds = "
    f"{rf_total_fits} entraînements"
)

print(
    f"HistGradientBoosting : "
    f"{HGB_N_ITER} configurations × "
    f"{CV_FOLDS} folds = "
    f"{hgb_total_fits} entraînements"
)

print(
    "\nMétrique principale : MAE"
)

print(
    "Scoring Scikit-learn : neg_mean_absolute_error"
)

print(
    "\n✅ Stratégie de tuning définie."
)

print(
    "✅ Aucun entraînement n'a encore été lancé."
)

print(
    "✅ Le Test final reste exclu du tuning."
)

,modele,configurations_testees,folds_cv,entrainements_totaux,metrique_selection
0,Random Forest,8,3,24,MAE
1,HistGradientBoosting,20,3,60,MAE



STRATÉGIE DE TUNING

Random Forest : 8 configurations × 3 folds = 24 entraînements
HistGradientBoosting : 20 configurations × 3 folds = 60 entraînements

Métrique principale : MAE
Scoring Scikit-learn : neg_mean_absolute_error

✅ Stratégie de tuning définie.
✅ Aucun entraînement n'a encore été lancé.
✅ Le Test final reste exclu du tuning.


### 8.4 Optimisation du Random Forest Regressor

### Objectif

L'objectif de cette étape est d'identifier une configuration plus performante
du `RandomForestRegressor` à partir de l'espace d'hyperparamètres défini
précédemment.

La recherche est réalisée avec `RandomizedSearchCV`.

Cette méthode teste un nombre limité de configurations tirées aléatoirement
dans l'espace défini.

Pour Random Forest :

- le nombre de configurations testées est défini par `RF_N_ITER` ;
- chaque configuration est évaluée avec une validation croisée à `CV_FOLDS`
  folds ;
- la métrique principale de sélection est la MAE ;
- le jeu `Test final` n'est pas utilisé pendant cette étape.

Le temps total nécessaire à la recherche est également mesuré afin d'évaluer
le coût réel du tuning.

À l'issue de cette étape, les meilleurs hyperparamètres et la meilleure MAE
obtenue en validation croisée sont conservés pour la suite.

In [12]:
# ---------------------------------------------------------------------
# 8.4 - Optimisation contrôlée du Random Forest avec RandomizedSearchCV
# ---------------------------------------------------------------------

import time

import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV


# ---------------------------------------------------------------------
# 1. Modèle Random Forest utilisé pendant le tuning
#
# n_jobs=-1 :
# un seul Random Forest à la fois peut utiliser les cœurs CPU disponibles.
# ---------------------------------------------------------------------

random_forest_tuning_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


# ---------------------------------------------------------------------
# 2. Définition de la recherche aléatoire
#
# IMPORTANT :
# n_jobs=1 au niveau de RandomizedSearchCV évite de lancer
# plusieurs Random Forest simultanément.
#
# Les configurations / folds sont donc traités séquentiellement,
# ce qui réduit fortement le risque de saturation mémoire.
# ---------------------------------------------------------------------

random_forest_search = RandomizedSearchCV(
    estimator=random_forest_tuning_model,
    param_distributions=random_forest_param_distributions,
    n_iter=RF_N_ITER,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True,
)


# ---------------------------------------------------------------------
# 3. Rappel de la charge avant lancement
# ---------------------------------------------------------------------

rf_total_fits = (
    RF_N_ITER
    * CV_FOLDS
)

print("OPTIMISATION RANDOM FOREST")
print("=" * 70)

print(
    f"\nObservations utilisées pour le tuning : "
    f"{len(X_tuning_train):,}"
)

print(
    f"Variables explicatives                 : "
    f"{X_tuning_train.shape[1]}"
)

print(
    f"Configurations testées                : "
    f"{RF_N_ITER}"
)

print(
    f"Validation croisée                    : "
    f"{CV_FOLDS} folds"
)

print(
    f"Entraînements CV prévus               : "
    f"{rf_total_fits}"
)

print(
    "\nMode d'exécution : séquentiel "
    "(une configuration/fold à la fois)"
)

print(
    "\nDébut du tuning Random Forest..."
)


# ---------------------------------------------------------------------
# 4. Lancement du tuning
# ---------------------------------------------------------------------

rf_tuning_start_time = time.perf_counter()

random_forest_search.fit(
    X_tuning_train,
    y_tuning_train,
)

rf_tuning_time_seconds = (
    time.perf_counter()
    - rf_tuning_start_time
)


# ---------------------------------------------------------------------
# 5. Récupération de la meilleure configuration
# ---------------------------------------------------------------------

rf_best_params = (
    random_forest_search.best_params_
)

rf_best_cv_mae = (
    -random_forest_search.best_score_
)

rf_best_model = (
    random_forest_search.best_estimator_
)


# ---------------------------------------------------------------------
# 6. Récupération des résultats de toutes les configurations testées
# ---------------------------------------------------------------------

rf_cv_results_df = pd.DataFrame(
    random_forest_search.cv_results_
)


# ---------------------------------------------------------------------
# 7. Conversion des scores négatifs en MAE positives
# ---------------------------------------------------------------------

rf_cv_results_df["mae_train_cv"] = (
    -rf_cv_results_df["mean_train_score"]
)

rf_cv_results_df["mae_validation_cv"] = (
    -rf_cv_results_df["mean_test_score"]
)

rf_cv_results_df["std_mae_validation_cv"] = (
    rf_cv_results_df["std_test_score"]
)


# ---------------------------------------------------------------------
# 8. Tableau de comparaison des configurations
# ---------------------------------------------------------------------

rf_tuning_results_df = (
    rf_cv_results_df[
        [
            "rank_test_score",
            "mae_train_cv",
            "mae_validation_cv",
            "std_mae_validation_cv",
            "mean_fit_time",
            "params",
        ]
    ]
    .sort_values(
        by="rank_test_score",
        ascending=True,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 9. Version arrondie destinée uniquement à l'affichage
# ---------------------------------------------------------------------

rf_tuning_results_display_df = (
    rf_tuning_results_df.copy()
)

columns_to_round = [
    "mae_train_cv",
    "mae_validation_cv",
    "std_mae_validation_cv",
    "mean_fit_time",
]

rf_tuning_results_display_df[
    columns_to_round
] = (
    rf_tuning_results_display_df[
        columns_to_round
    ]
    .round(4)
)


# ---------------------------------------------------------------------
# 10. Rapport final du tuning
# ---------------------------------------------------------------------

print("\n")
print("RÉSULTATS DU TUNING RANDOM FOREST")
print("=" * 70)

print(
    f"\nMeilleure MAE en validation croisée : "
    f"{rf_best_cv_mae:.4f} g CO₂/km"
)

print(
    f"Temps total du tuning                : "
    f"{rf_tuning_time_seconds:.2f} secondes"
)

print(
    "\nMeilleurs hyperparamètres :"
)

for parameter, value in rf_best_params.items():
    print(
        f"  - {parameter}: {value}"
    )


print(
    "\nClassement des configurations testées :"
)

display(
    rf_tuning_results_display_df
)


print(
    "\n✅ Optimisation Random Forest terminée."
)

print(
    "✅ La meilleure configuration est conservée "
    "dans 'rf_best_model'."
)

print(
    "✅ Le Test final n'a pas été utilisé pendant le tuning."
)

OPTIMISATION RANDOM FOREST

Observations utilisées pour le tuning : 400,000
Variables explicatives                 : 41
Configurations testées                : 8
Validation croisée                    : 3 folds
Entraînements CV prévus               : 24

Mode d'exécution : séquentiel (une configuration/fold à la fois)

Début du tuning Random Forest...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.2min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.1min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=5, min_samples_split=10, n_estimators=100; total time= 1.0min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 1.2min
[CV] END max_depth=20, max_features=0.7, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total ti

,rank_test_score,mae_train_cv,mae_validation_cv,std_mae_validation_cv,mean_fit_time,params
0,1,0.2723,0.4708,0.0011,91.0534,"{'n_estimators': 100, 'min_samples_split': 5, ..."
1,2,0.3415,0.4928,0.0055,131.5336,"{'n_estimators': 200, 'min_samples_split': 10,..."
2,3,0.3407,0.5176,0.0023,66.0007,"{'n_estimators': 100, 'min_samples_split': 2, ..."
3,4,0.3438,0.5313,0.0016,282.3124,"{'n_estimators': 300, 'min_samples_split': 2, ..."
4,5,0.5238,0.6277,0.0067,64.7831,"{'n_estimators': 100, 'min_samples_split': 2, ..."
5,5,0.5238,0.6277,0.0067,64.6074,"{'n_estimators': 100, 'min_samples_split': 10,..."
6,7,1.1992,1.2935,0.0315,23.8786,"{'n_estimators': 100, 'min_samples_split': 2, ..."
7,8,1.7115,1.7906,0.0278,19.6929,"{'n_estimators': 100, 'min_samples_split': 2, ..."



✅ Optimisation Random Forest terminée.
✅ La meilleure configuration est conservée dans 'rf_best_model'.
✅ Le Test final n'a pas été utilisé pendant le tuning.


### 8.5 Validation du Random Forest optimisé

### Objectif

La recherche d'hyperparamètres réalisée précédemment a identifié la meilleure
configuration du Random Forest à partir d'une validation croisée effectuée
exclusivement sur le jeu d'apprentissage du tuning.

La configuration sélectionnée doit maintenant être évaluée sur le jeu
`Validation tuning`, constitué de données qui n'ont pas participé à la
recherche des hyperparamètres.

Cette étape permet de vérifier que les performances observées pendant la
validation croisée se maintiennent sur un ensemble indépendant.

L'évaluation repose sur les trois métriques utilisées précédemment :

- **MAE** : erreur absolue moyenne ;
- **RMSE** : racine de l'erreur quadratique moyenne ;
- **R²** : proportion de la variance de la cible expliquée par le modèle.

Les résultats obtenus seront également comparés à la MAE moyenne issue de la
validation croisée afin d'évaluer la stabilité de la configuration retenue.

Le jeu `Test final` reste totalement exclu de cette étape.

In [14]:
# ---------------------------------------------------------------------
# 8.5 - Validation du Random Forest optimisé
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ---------------------------------------------------------------------
# 1. Prédictions sur le jeu Validation tuning
#
# rf_best_model a déjà été réentraîné automatiquement par
# RandomizedSearchCV grâce à refit=True sur l'ensemble de
# X_tuning_train.
#
# Aucun nouvel entraînement n'est donc nécessaire ici.
# ---------------------------------------------------------------------

y_rf_tuning_validation_pred = rf_best_model.predict(
    X_tuning_validation
)


# ---------------------------------------------------------------------
# 2. Calcul des métriques sur Validation tuning
# ---------------------------------------------------------------------

rf_validation_mae = mean_absolute_error(
    y_tuning_validation,
    y_rf_tuning_validation_pred,
)

rf_validation_rmse = np.sqrt(
    mean_squared_error(
        y_tuning_validation,
        y_rf_tuning_validation_pred,
    )
)

rf_validation_r2 = r2_score(
    y_tuning_validation,
    y_rf_tuning_validation_pred,
)


# ---------------------------------------------------------------------
# 3. Comparaison avec la MAE obtenue en validation croisée
#
# Écart signé :
#   > 0 : la MAE augmente sur Validation tuning -> dégradation
#   < 0 : la MAE diminue sur Validation tuning -> amélioration
# ---------------------------------------------------------------------

rf_cv_validation_gap = (
    rf_validation_mae
    - rf_best_cv_mae
)

rf_cv_validation_gap_abs = abs(
    rf_cv_validation_gap
)

rf_cv_validation_gap_percent = (
    rf_cv_validation_gap_abs
    / rf_best_cv_mae
    * 100
)


# ---------------------------------------------------------------------
# 4. Tableau récapitulatif
#
# La CV fournit ici la MAE utilisée pour la sélection.
# RMSE et R² ne sont pas renseignés pour cette ligne car le tuning
# a été réalisé avec neg_mean_absolute_error comme scoring principal.
# ---------------------------------------------------------------------

rf_optimized_validation_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest optimisé",
            "jeu_evaluation": "Validation croisée (CV)",
            "observations": len(X_tuning_train),
            "MAE_g_co2_km": rf_best_cv_mae,
            "RMSE_g_co2_km": np.nan,
            "R2": np.nan,
        },
        {
            "modele": "Random Forest optimisé",
            "jeu_evaluation": "Validation tuning indépendante",
            "observations": len(X_tuning_validation),
            "MAE_g_co2_km": rf_validation_mae,
            "RMSE_g_co2_km": rf_validation_rmse,
            "R2": rf_validation_r2,
        },
    ]
)


# ---------------------------------------------------------------------
# 5. Version destinée uniquement à l'affichage
# ---------------------------------------------------------------------

rf_optimized_validation_display_df = (
    rf_optimized_validation_df.copy()
)

metric_columns = [
    "MAE_g_co2_km",
    "RMSE_g_co2_km",
    "R2",
]

rf_optimized_validation_display_df[
    metric_columns
] = (
    rf_optimized_validation_display_df[
        metric_columns
    ]
    .round(4)
)

display(
    rf_optimized_validation_display_df
)


# ---------------------------------------------------------------------
# 6. Rapport des résultats
# ---------------------------------------------------------------------

print("\nVALIDATION DU RANDOM FOREST OPTIMISÉ")
print("=" * 70)

print(
    f"\nMAE moyenne obtenue en CV       : "
    f"{rf_best_cv_mae:.4f} g CO₂/km"
)

print(
    f"MAE Validation tuning           : "
    f"{rf_validation_mae:.4f} g CO₂/km"
)

print(
    f"RMSE Validation tuning          : "
    f"{rf_validation_rmse:.4f} g CO₂/km"
)

print(
    f"R² Validation tuning            : "
    f"{rf_validation_r2:.4f}"
)

print(
    f"\nÉcart MAE Validation - CV       : "
    f"{rf_cv_validation_gap:+.4f} g CO₂/km"
)

print(
    f"Amplitude relative de l'écart   : "
    f"{rf_cv_validation_gap_percent:.2f} %"
)


# ---------------------------------------------------------------------
# 7. Interprétation dynamique de la généralisation
# ---------------------------------------------------------------------

print("\nINTERPRÉTATION")
print("-" * 70)


# ---------------------------------------------------------------------
# CAS 1 : amélioration sur Validation tuning
# ---------------------------------------------------------------------

if rf_cv_validation_gap < 0:

    print(
        "✅ La MAE obtenue sur Validation tuning est inférieure "
        "à la MAE moyenne obtenue en validation croisée."
    )

    print(
        f"   L'amélioration observée est de "
        f"{rf_cv_validation_gap_abs:.4f} g CO₂/km "
        f"({rf_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   Aucun signal de dégradation de généralisation "
        "n'est observé à cette étape."
    )

    print(
        "   La configuration Random Forest optimisée est donc "
        "validée sur ce jeu indépendant du tuning."
    )


# ---------------------------------------------------------------------
# CAS 2 : performances pratiquement identiques
# ---------------------------------------------------------------------

elif rf_cv_validation_gap_percent <= 5:

    print(
        "✅ La MAE obtenue sur Validation tuning est très proche "
        "de la MAE moyenne obtenue en validation croisée."
    )

    print(
        "   La configuration optimisée présente une bonne "
        "stabilité sur le jeu de validation indépendant."
    )


# ---------------------------------------------------------------------
# CAS 3 : dégradation modérée
# ---------------------------------------------------------------------

elif rf_cv_validation_gap_percent <= 10:

    print(
        "⚠️ La MAE augmente modérément sur le jeu "
        "Validation tuning."
    )

    print(
        f"   La dégradation observée est de "
        f"{rf_cv_validation_gap:.4f} g CO₂/km "
        f"({rf_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   La configuration reste exploitable, mais sa "
        "généralisation devra être surveillée."
    )


# ---------------------------------------------------------------------
# CAS 4 : dégradation importante
# ---------------------------------------------------------------------

else:

    print(
        "⚠️ La MAE augmente sensiblement sur le jeu "
        "Validation tuning."
    )

    print(
        f"   La dégradation observée est de "
        f"{rf_cv_validation_gap:.4f} g CO₂/km "
        f"({rf_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   La capacité de généralisation de cette configuration "
        "doit être réexaminée avant toute sélection finale."
    )


# ---------------------------------------------------------------------
# 8. Contrôle de non-utilisation du Test final
# ---------------------------------------------------------------------

print(
    "\n✅ Le Test final n'a pas été utilisé pendant "
    "l'optimisation ni cette validation."
)

,modele,jeu_evaluation,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Random Forest optimisé,Validation croisée (CV),400000,0.4708,NaN,NaN
1,Random Forest optimisé,Validation tuning indépendante,100000,0.4242,2.8,0.9977



VALIDATION DU RANDOM FOREST OPTIMISÉ

MAE moyenne obtenue en CV       : 0.4708 g CO₂/km
MAE Validation tuning           : 0.4242 g CO₂/km
RMSE Validation tuning          : 2.8000 g CO₂/km
R² Validation tuning            : 0.9977

Écart MAE Validation - CV       : -0.0466 g CO₂/km
Amplitude relative de l'écart   : 9.89 %

INTERPRÉTATION
----------------------------------------------------------------------
✅ La MAE obtenue sur Validation tuning est inférieure à la MAE moyenne obtenue en validation croisée.
   L'amélioration observée est de 0.0466 g CO₂/km (9.89 %).
   Aucun signal de dégradation de généralisation n'est observé à cette étape.
   La configuration Random Forest optimisée est donc validée sur ce jeu indépendant du tuning.

✅ Le Test final n'a pas été utilisé pendant l'optimisation ni cette validation.


### 8.6 Optimisation du HistGradientBoostingRegressor

### Objectif

L'objectif de cette étape est d'identifier une configuration optimisée du
`HistGradientBoostingRegressor` à partir de l'espace d'hyperparamètres défini
précédemment.

La recherche est réalisée avec `RandomizedSearchCV`.

Pour HistGradientBoosting :

- le nombre de configurations testées est défini par `HGB_N_ITER` ;
- chaque configuration est évaluée avec une validation croisée à `CV_FOLDS`
  folds ;
- la métrique principale de sélection est la MAE ;
- le jeu `Test final` n'est pas utilisé pendant cette étape.

Le temps total du tuning est mesuré afin de comparer le coût d'optimisation
avec celui du Random Forest.

À l'issue de cette étape, la meilleure configuration sera conservée puis
évaluée séparément sur le jeu `Validation tuning`.

In [15]:
# ---------------------------------------------------------------------
# 8.6 - Optimisation du HistGradientBoosting avec RandomizedSearchCV
# ---------------------------------------------------------------------

import time

import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV


# ---------------------------------------------------------------------
# 1. Modèle de base utilisé pendant le tuning
# ---------------------------------------------------------------------

hist_gradient_boosting_tuning_model = (
    HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
    )
)


# ---------------------------------------------------------------------
# 2. Définition de la recherche aléatoire
#
# n_jobs=1 :
# les configurations/folds sont exécutés séquentiellement afin de
# limiter la consommation simultanée de ressources.
# ---------------------------------------------------------------------

hist_gradient_boosting_search = RandomizedSearchCV(
    estimator=hist_gradient_boosting_tuning_model,
    param_distributions=hist_gradient_boosting_param_distributions,
    n_iter=HGB_N_ITER,
    scoring=SCORING,
    cv=CV_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=2,
    return_train_score=True,
    refit=True,
)


# ---------------------------------------------------------------------
# 3. Rappel de la charge avant lancement
# ---------------------------------------------------------------------

hgb_total_fits = (
    HGB_N_ITER
    * CV_FOLDS
)

print("OPTIMISATION HISTGRADIENTBOOSTING")
print("=" * 70)

print(
    f"\nObservations utilisées pour le tuning : "
    f"{len(X_tuning_train):,}"
)

print(
    f"Variables explicatives                 : "
    f"{X_tuning_train.shape[1]}"
)

print(
    f"Configurations testées                : "
    f"{HGB_N_ITER}"
)

print(
    f"Validation croisée                    : "
    f"{CV_FOLDS} folds"
)

print(
    f"Entraînements CV prévus               : "
    f"{hgb_total_fits}"
)

print(
    "\nMode d'exécution : séquentiel "
    "(une configuration/fold à la fois)"
)

print(
    "\nDébut du tuning HistGradientBoosting..."
)


# ---------------------------------------------------------------------
# 4. Lancement du tuning
# ---------------------------------------------------------------------

hgb_tuning_start_time = time.perf_counter()

hist_gradient_boosting_search.fit(
    X_tuning_train,
    y_tuning_train,
)

hgb_tuning_time_seconds = (
    time.perf_counter()
    - hgb_tuning_start_time
)


# ---------------------------------------------------------------------
# 5. Meilleure configuration
# ---------------------------------------------------------------------

hgb_best_params = (
    hist_gradient_boosting_search.best_params_
)

hgb_best_cv_mae = (
    -hist_gradient_boosting_search.best_score_
)

hgb_best_model = (
    hist_gradient_boosting_search.best_estimator_
)


# ---------------------------------------------------------------------
# 6. Récupération des résultats
# ---------------------------------------------------------------------

hgb_cv_results_df = pd.DataFrame(
    hist_gradient_boosting_search.cv_results_
)


# ---------------------------------------------------------------------
# 7. Conversion des scores négatifs en MAE positives
# ---------------------------------------------------------------------

hgb_cv_results_df["mae_train_cv"] = (
    -hgb_cv_results_df["mean_train_score"]
)

hgb_cv_results_df["mae_validation_cv"] = (
    -hgb_cv_results_df["mean_test_score"]
)

hgb_cv_results_df["std_mae_validation_cv"] = (
    hgb_cv_results_df["std_test_score"]
)


# ---------------------------------------------------------------------
# 8. Tableau de comparaison des configurations
# ---------------------------------------------------------------------

hgb_tuning_results_df = (
    hgb_cv_results_df[
        [
            "rank_test_score",
            "mae_train_cv",
            "mae_validation_cv",
            "std_mae_validation_cv",
            "mean_fit_time",
            "params",
        ]
    ]
    .sort_values(
        by="rank_test_score",
        ascending=True,
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 9. Version arrondie pour l'affichage
# ---------------------------------------------------------------------

hgb_tuning_results_display_df = (
    hgb_tuning_results_df.copy()
)

columns_to_round = [
    "mae_train_cv",
    "mae_validation_cv",
    "std_mae_validation_cv",
    "mean_fit_time",
]

hgb_tuning_results_display_df[
    columns_to_round
] = (
    hgb_tuning_results_display_df[
        columns_to_round
    ]
    .round(4)
)


# ---------------------------------------------------------------------
# 10. Rapport final du tuning
# ---------------------------------------------------------------------

print("\n")
print("RÉSULTATS DU TUNING HISTGRADIENTBOOSTING")
print("=" * 70)

print(
    f"\nMeilleure MAE en validation croisée : "
    f"{hgb_best_cv_mae:.4f} g CO₂/km"
)

print(
    f"Temps total du tuning                : "
    f"{hgb_tuning_time_seconds:.2f} secondes"
)

print(
    "\nMeilleurs hyperparamètres :"
)

for parameter, value in hgb_best_params.items():
    print(
        f"  - {parameter}: {value}"
    )


print(
    "\nClassement des configurations testées :"
)

display(
    hgb_tuning_results_display_df
)


print(
    "\n✅ Optimisation HistGradientBoosting terminée."
)

print(
    "✅ La meilleure configuration est conservée "
    "dans 'hgb_best_model'."
)

print(
    "✅ Le Test final n'a pas été utilisé pendant le tuning."
)

OPTIMISATION HISTGRADIENTBOOSTING

Observations utilisées pour le tuning : 400,000
Variables explicatives                 : 41
Configurations testées                : 20
Validation croisée                    : 3 folds
Entraînements CV prévus               : 60

Mode d'exécution : séquentiel (une configuration/fold à la fois)

Début du tuning HistGradientBoosting...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  32.1s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  22.6s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=300, max_leaf_nodes=63, min_samples_leaf=20; total time=  26.0s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_leaf_nodes=15, min_samples_leaf=20; total time=   8.4s
[CV] END l2_regularization=0.1, learning_rate=0.05, max_iter=100, max_l

,rank_test_score,mae_train_cv,mae_validation_cv,std_mae_validation_cv,mean_fit_time,params
0,1,1.1585,1.2192,0.0084,24.9470,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63,..."
1,2,1.2551,1.2954,0.0132,27.5902,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
2,3,1.3180,1.3577,0.0099,18.4538,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
3,4,1.3350,1.3816,0.0099,24.7067,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63,..."
4,5,1.3449,1.3825,0.0105,26.4597,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
5,6,1.5449,1.5701,0.0103,22.9531,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
6,7,1.6743,1.7004,0.0102,15.2470,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63,..."
7,8,1.6891,1.7112,0.0143,11.5086,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63..."
8,9,1.7613,1.7820,0.0081,7.5219,"{'min_samples_leaf': 100, 'max_leaf_nodes': 31..."
9,10,1.7632,1.7966,0.0047,10.2074,"{'min_samples_leaf': 20, 'max_leaf_nodes': 31,..."



✅ Optimisation HistGradientBoosting terminée.
✅ La meilleure configuration est conservée dans 'hgb_best_model'.
✅ Le Test final n'a pas été utilisé pendant le tuning.


### 8.7 Validation du HistGradientBoosting optimisé

### Objectif

La recherche d'hyperparamètres réalisée précédemment a identifié la meilleure
configuration du `HistGradientBoostingRegressor` à partir d'une validation
croisée effectuée exclusivement sur le jeu d'apprentissage du tuning.

La configuration sélectionnée doit maintenant être évaluée sur le jeu
`Validation tuning`, constitué de 100 000 observations qui n'ont pas participé
à la recherche des hyperparamètres.

Cette étape permet de vérifier que les performances observées pendant la
validation croisée se maintiennent sur un ensemble indépendant.

L'évaluation repose sur les trois métriques utilisées précédemment :

- **MAE** : erreur absolue moyenne ;
- **RMSE** : racine de l'erreur quadratique moyenne ;
- **R²** : proportion de la variance de la cible expliquée par le modèle.

La MAE obtenue sur le jeu de validation indépendant est également comparée à
la MAE moyenne obtenue pendant la validation croisée.

L'écart est interprété en tenant compte de son signe :

- un écart négatif correspond à une amélioration de la MAE ;
- un écart positif correspond à une dégradation de la MAE.

Le jeu `Test final` reste totalement exclu de cette étape.

In [16]:
# ---------------------------------------------------------------------
# 8.7 - Validation du HistGradientBoosting optimisé
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ---------------------------------------------------------------------
# 1. Prédictions sur le jeu Validation tuning
#
# hgb_best_model a déjà été réentraîné automatiquement par
# RandomizedSearchCV grâce à refit=True sur l'ensemble de
# X_tuning_train.
#
# Aucun nouvel entraînement n'est donc nécessaire ici.
# ---------------------------------------------------------------------

y_hgb_tuning_validation_pred = hgb_best_model.predict(
    X_tuning_validation
)


# ---------------------------------------------------------------------
# 2. Calcul des métriques sur Validation tuning
# ---------------------------------------------------------------------

hgb_validation_mae = mean_absolute_error(
    y_tuning_validation,
    y_hgb_tuning_validation_pred,
)

hgb_validation_rmse = np.sqrt(
    mean_squared_error(
        y_tuning_validation,
        y_hgb_tuning_validation_pred,
    )
)

hgb_validation_r2 = r2_score(
    y_tuning_validation,
    y_hgb_tuning_validation_pred,
)


# ---------------------------------------------------------------------
# 3. Comparaison avec la MAE obtenue en validation croisée
#
# Écart signé :
#   > 0 : la MAE augmente sur Validation tuning -> dégradation
#   < 0 : la MAE diminue sur Validation tuning -> amélioration
# ---------------------------------------------------------------------

hgb_cv_validation_gap = (
    hgb_validation_mae
    - hgb_best_cv_mae
)

hgb_cv_validation_gap_abs = abs(
    hgb_cv_validation_gap
)

hgb_cv_validation_gap_percent = (
    hgb_cv_validation_gap_abs
    / hgb_best_cv_mae
    * 100
)


# ---------------------------------------------------------------------
# 4. Tableau récapitulatif
#
# La CV fournit ici la MAE utilisée pour la sélection.
# RMSE et R² ne sont pas renseignés pour cette ligne car le tuning
# a été réalisé avec neg_mean_absolute_error comme scoring principal.
# ---------------------------------------------------------------------

hgb_optimized_validation_df = pd.DataFrame(
    [
        {
            "modele": "HistGradientBoosting optimisé",
            "jeu_evaluation": "Validation croisée (CV)",
            "observations": len(X_tuning_train),
            "MAE_g_co2_km": hgb_best_cv_mae,
            "RMSE_g_co2_km": np.nan,
            "R2": np.nan,
        },
        {
            "modele": "HistGradientBoosting optimisé",
            "jeu_evaluation": "Validation tuning indépendante",
            "observations": len(X_tuning_validation),
            "MAE_g_co2_km": hgb_validation_mae,
            "RMSE_g_co2_km": hgb_validation_rmse,
            "R2": hgb_validation_r2,
        },
    ]
)


# ---------------------------------------------------------------------
# 5. Version destinée uniquement à l'affichage
# ---------------------------------------------------------------------

hgb_optimized_validation_display_df = (
    hgb_optimized_validation_df.copy()
)

metric_columns = [
    "MAE_g_co2_km",
    "RMSE_g_co2_km",
    "R2",
]

hgb_optimized_validation_display_df[
    metric_columns
] = (
    hgb_optimized_validation_display_df[
        metric_columns
    ]
    .round(4)
)

display(
    hgb_optimized_validation_display_df
)


# ---------------------------------------------------------------------
# 6. Rapport des résultats
# ---------------------------------------------------------------------

print("\nVALIDATION DU HISTGRADIENTBOOSTING OPTIMISÉ")
print("=" * 70)

print(
    f"\nMAE moyenne obtenue en CV       : "
    f"{hgb_best_cv_mae:.4f} g CO₂/km"
)

print(
    f"MAE Validation tuning           : "
    f"{hgb_validation_mae:.4f} g CO₂/km"
)

print(
    f"RMSE Validation tuning          : "
    f"{hgb_validation_rmse:.4f} g CO₂/km"
)

print(
    f"R² Validation tuning            : "
    f"{hgb_validation_r2:.4f}"
)

print(
    f"\nÉcart MAE Validation - CV       : "
    f"{hgb_cv_validation_gap:+.4f} g CO₂/km"
)

print(
    f"Amplitude relative de l'écart   : "
    f"{hgb_cv_validation_gap_percent:.2f} %"
)


# ---------------------------------------------------------------------
# 7. Interprétation dynamique de la généralisation
# ---------------------------------------------------------------------

print("\nINTERPRÉTATION")
print("-" * 70)


# ---------------------------------------------------------------------
# CAS 1 : amélioration sur Validation tuning
# ---------------------------------------------------------------------

if hgb_cv_validation_gap < 0:

    print(
        "✅ La MAE obtenue sur Validation tuning est inférieure "
        "à la MAE moyenne obtenue en validation croisée."
    )

    print(
        f"   L'amélioration observée est de "
        f"{hgb_cv_validation_gap_abs:.4f} g CO₂/km "
        f"({hgb_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   Aucun signal de dégradation de généralisation "
        "n'est observé à cette étape."
    )

    print(
        "   La configuration HistGradientBoosting optimisée est "
        "donc validée sur ce jeu indépendant du tuning."
    )


# ---------------------------------------------------------------------
# CAS 2 : performances pratiquement identiques
# ---------------------------------------------------------------------

elif hgb_cv_validation_gap_percent <= 5:

    print(
        "✅ La MAE obtenue sur Validation tuning est très proche "
        "de la MAE moyenne obtenue en validation croisée."
    )

    print(
        "   La configuration optimisée présente une bonne "
        "stabilité sur le jeu de validation indépendant."
    )


# ---------------------------------------------------------------------
# CAS 3 : dégradation modérée
# ---------------------------------------------------------------------

elif hgb_cv_validation_gap_percent <= 10:

    print(
        "⚠️ La MAE augmente modérément sur le jeu "
        "Validation tuning."
    )

    print(
        f"   La dégradation observée est de "
        f"{hgb_cv_validation_gap:.4f} g CO₂/km "
        f"({hgb_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   La configuration reste exploitable, mais sa "
        "généralisation devra être surveillée."
    )


# ---------------------------------------------------------------------
# CAS 4 : dégradation importante
# ---------------------------------------------------------------------

else:

    print(
        "⚠️ La MAE augmente sensiblement sur le jeu "
        "Validation tuning."
    )

    print(
        f"   La dégradation observée est de "
        f"{hgb_cv_validation_gap:.4f} g CO₂/km "
        f"({hgb_cv_validation_gap_percent:.2f} %)."
    )

    print(
        "   La capacité de généralisation de cette configuration "
        "doit être réexaminée avant toute sélection finale."
    )


# ---------------------------------------------------------------------
# 8. Contrôle de non-utilisation du Test final
# ---------------------------------------------------------------------

print(
    "\n✅ Le Test final n'a pas été utilisé pendant "
    "l'optimisation ni cette validation."
)

,modele,jeu_evaluation,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,HistGradientBoosting optimisé,Validation croisée (CV),400000,1.2192,NaN,NaN
1,HistGradientBoosting optimisé,Validation tuning indépendante,100000,1.2203,3.6587,0.996



VALIDATION DU HISTGRADIENTBOOSTING OPTIMISÉ

MAE moyenne obtenue en CV       : 1.2192 g CO₂/km
MAE Validation tuning           : 1.2203 g CO₂/km
RMSE Validation tuning          : 3.6587 g CO₂/km
R² Validation tuning            : 0.9960

Écart MAE Validation - CV       : +0.0011 g CO₂/km
Amplitude relative de l'écart   : 0.09 %

INTERPRÉTATION
----------------------------------------------------------------------
✅ La MAE obtenue sur Validation tuning est très proche de la MAE moyenne obtenue en validation croisée.
   La configuration optimisée présente une bonne stabilité sur le jeu de validation indépendant.

✅ Le Test final n'a pas été utilisé pendant l'optimisation ni cette validation.


### 8.8 Comparaison des modèles optimisés

### Objectif

Les deux modèles candidats ont maintenant été optimisés selon le même protocole
et évalués sur le même jeu `Validation tuning`, indépendant des données utilisées
pendant la recherche des hyperparamètres.

Cette étape vise à comparer les deux modèles selon plusieurs dimensions :

- la **MAE** sur le jeu de validation indépendant ;
- la **RMSE** sur le jeu de validation indépendant ;
- le **R²** sur le jeu de validation indépendant ;
- la stabilité entre validation croisée et validation indépendante ;
- le temps nécessaire à l'optimisation ;
- le nombre de configurations et d'entraînements réalisés.

La comparaison ne doit pas reposer sur une seule métrique.

Un modèle peut présenter une meilleure performance prédictive tout en étant
plus coûteux à entraîner, tandis qu'un autre peut présenter une stabilité
supérieure et un coût computationnel inférieur.

Le choix du candidat final doit donc expliciter le compromis entre :

1. **performance prédictive** ;
2. **stabilité de généralisation** ;
3. **coût computationnel**.

Le jeu `Test final` reste exclu de cette comparaison. Il sera utilisé seulement
après la sélection du candidat retenu.

In [17]:
# ---------------------------------------------------------------------
# 8.8 - Comparaison des modèles optimisés
# ---------------------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------------------
# 1. Construction du tableau comparatif
# ---------------------------------------------------------------------

optimized_models_comparison_df = pd.DataFrame(
    [
        {
            "modele": "Random Forest",
            "mae_cv": rf_best_cv_mae,
            "mae_validation": rf_validation_mae,
            "rmse_validation": rf_validation_rmse,
            "r2_validation": rf_validation_r2,
            "ecart_mae_cv_validation": rf_cv_validation_gap,
            "ecart_mae_relatif_pct": rf_cv_validation_gap_percent,
            "temps_tuning_secondes": rf_tuning_time_seconds,
            "configurations_testees": RF_N_ITER,
            "fits_cv": RF_N_ITER * CV_FOLDS,
        },
        {
            "modele": "HistGradientBoosting",
            "mae_cv": hgb_best_cv_mae,
            "mae_validation": hgb_validation_mae,
            "rmse_validation": hgb_validation_rmse,
            "r2_validation": hgb_validation_r2,
            "ecart_mae_cv_validation": hgb_cv_validation_gap,
            "ecart_mae_relatif_pct": hgb_cv_validation_gap_percent,
            "temps_tuning_secondes": hgb_tuning_time_seconds,
            "configurations_testees": HGB_N_ITER,
            "fits_cv": HGB_N_ITER * CV_FOLDS,
        },
    ]
)


# ---------------------------------------------------------------------
# 2. Calcul du temps de tuning en minutes
# ---------------------------------------------------------------------

optimized_models_comparison_df[
    "temps_tuning_minutes"
] = (
    optimized_models_comparison_df[
        "temps_tuning_secondes"
    ]
    / 60
)


# ---------------------------------------------------------------------
# 3. Organisation des colonnes
# ---------------------------------------------------------------------

optimized_models_comparison_df = (
    optimized_models_comparison_df[
        [
            "modele",
            "mae_cv",
            "mae_validation",
            "rmse_validation",
            "r2_validation",
            "ecart_mae_cv_validation",
            "ecart_mae_relatif_pct",
            "temps_tuning_minutes",
            "configurations_testees",
            "fits_cv",
        ]
    ]
)


# ---------------------------------------------------------------------
# 4. Version arrondie destinée à l'affichage
# ---------------------------------------------------------------------

optimized_models_comparison_display_df = (
    optimized_models_comparison_df.copy()
)

columns_to_round = [
    "mae_cv",
    "mae_validation",
    "rmse_validation",
    "r2_validation",
    "ecart_mae_cv_validation",
    "ecart_mae_relatif_pct",
    "temps_tuning_minutes",
]

optimized_models_comparison_display_df[
    columns_to_round
] = (
    optimized_models_comparison_display_df[
        columns_to_round
    ]
    .round(4)
)


display(
    optimized_models_comparison_display_df
)


# ---------------------------------------------------------------------
# 5. Identification automatique des meilleurs résultats
# ---------------------------------------------------------------------

best_mae_model = (
    optimized_models_comparison_df
    .loc[
        optimized_models_comparison_df[
            "mae_validation"
        ].idxmin()
    ]
)

best_rmse_model = (
    optimized_models_comparison_df
    .loc[
        optimized_models_comparison_df[
            "rmse_validation"
        ].idxmin()
    ]
)

best_r2_model = (
    optimized_models_comparison_df
    .loc[
        optimized_models_comparison_df[
            "r2_validation"
        ].idxmax()
    ]
)

best_stability_model = (
    optimized_models_comparison_df
    .loc[
        optimized_models_comparison_df[
            "ecart_mae_relatif_pct"
        ].idxmin()
    ]
)

fastest_tuning_model = (
    optimized_models_comparison_df
    .loc[
        optimized_models_comparison_df[
            "temps_tuning_minutes"
        ].idxmin()
    ]
)


# ---------------------------------------------------------------------
# 6. Rapport comparatif
# ---------------------------------------------------------------------

print("\nCOMPARAISON DES MODÈLES OPTIMISÉS")
print("=" * 70)


print("\n1. Meilleure MAE sur Validation tuning")
print(
    f"   {best_mae_model['modele']} : "
    f"{best_mae_model['mae_validation']:.4f} g CO₂/km"
)


print("\n2. Meilleure RMSE sur Validation tuning")
print(
    f"   {best_rmse_model['modele']} : "
    f"{best_rmse_model['rmse_validation']:.4f} g CO₂/km"
)


print("\n3. Meilleur R² sur Validation tuning")
print(
    f"   {best_r2_model['modele']} : "
    f"{best_r2_model['r2_validation']:.4f}"
)


print("\n4. Meilleure stabilité CV → Validation")
print(
    f"   {best_stability_model['modele']} : "
    f"{best_stability_model['ecart_mae_relatif_pct']:.2f} %"
)


print("\n5. Temps de tuning le plus faible")
print(
    f"   {fastest_tuning_model['modele']} : "
    f"{fastest_tuning_model['temps_tuning_minutes']:.2f} minutes"
)


# ---------------------------------------------------------------------
# 7. Comparaison directe des performances prédictives
# ---------------------------------------------------------------------

rf_vs_hgb_mae_difference = (
    hgb_validation_mae
    - rf_validation_mae
)

rf_vs_hgb_rmse_difference = (
    hgb_validation_rmse
    - rf_validation_rmse
)

rf_vs_hgb_mae_ratio = (
    hgb_validation_mae
    / rf_validation_mae
)

rf_vs_hgb_tuning_time_ratio = (
    rf_tuning_time_seconds
    / hgb_tuning_time_seconds
)


print("\n6. Comparaison directe")
print("-" * 70)

print(
    f"\nÉcart de MAE en faveur de Random Forest : "
    f"{rf_vs_hgb_mae_difference:.4f} g CO₂/km"
)

print(
    f"Écart de RMSE en faveur de Random Forest : "
    f"{rf_vs_hgb_rmse_difference:.4f} g CO₂/km"
)

print(
    f"MAE HistGradientBoosting / Random Forest : "
    f"{rf_vs_hgb_mae_ratio:.2f} ×"
)

print(
    f"Temps tuning Random Forest / "
    f"HistGradientBoosting : "
    f"{rf_vs_hgb_tuning_time_ratio:.2f} ×"
)


# ---------------------------------------------------------------------
# 8. Conclusion multicritère
# ---------------------------------------------------------------------

print("\nCONCLUSION MULTICRITÈRE")
print("=" * 70)

print(
    "\nRandom Forest présente la meilleure performance prédictive "
    "sur le jeu Validation tuning :"
)

print(
    f"  - MAE  : {rf_validation_mae:.4f} g CO₂/km"
)

print(
    f"  - RMSE : {rf_validation_rmse:.4f} g CO₂/km"
)

print(
    f"  - R²   : {rf_validation_r2:.4f}"
)


print(
    "\nHistGradientBoosting présente une meilleure stabilité "
    "entre validation croisée et validation indépendante :"
)

print(
    f"  - écart relatif MAE : "
    f"{hgb_cv_validation_gap_percent:.2f} %"
)

print(
    f"  - temps de tuning   : "
    f"{hgb_tuning_time_seconds / 60:.2f} minutes"
)


print(
    "\nRandom Forest présente cependant une MAE Validation "
    f"{rf_vs_hgb_mae_ratio:.2f} fois plus faible que "
    "HistGradientBoosting."
)

print(
    "\nLe coût computationnel supérieur du Random Forest doit "
    "donc être mis en balance avec son avantage important en "
    "performance prédictive."
)

print(
    "\n✅ Les deux modèles ont été comparés sur le même jeu "
    "Validation tuning."
)

print(
    "✅ Le Test final n'a pas été utilisé pour cette comparaison."
)

,modele,mae_cv,mae_validation,rmse_validation,r2_validation,ecart_mae_cv_validation,ecart_mae_relatif_pct,temps_tuning_minutes,configurations_testees,fits_cv
0,Random Forest,0.4708,0.4242,2.8000,0.9977,-0.0466,9.8922,39.0823,8,24
1,HistGradientBoosting,1.2192,1.2203,3.6587,0.9960,0.0011,0.0932,18.7008,20,60



COMPARAISON DES MODÈLES OPTIMISÉS

1. Meilleure MAE sur Validation tuning
   Random Forest : 0.4242 g CO₂/km

2. Meilleure RMSE sur Validation tuning
   Random Forest : 2.8000 g CO₂/km

3. Meilleur R² sur Validation tuning
   Random Forest : 0.9977

4. Meilleure stabilité CV → Validation
   HistGradientBoosting : 0.09 %

5. Temps de tuning le plus faible
   HistGradientBoosting : 18.70 minutes

6. Comparaison directe
----------------------------------------------------------------------

Écart de MAE en faveur de Random Forest : 0.7961 g CO₂/km
Écart de RMSE en faveur de Random Forest : 0.8587 g CO₂/km
MAE HistGradientBoosting / Random Forest : 2.88 ×
Temps tuning Random Forest / HistGradientBoosting : 2.09 ×

CONCLUSION MULTICRITÈRE

Random Forest présente la meilleure performance prédictive sur le jeu Validation tuning :
  - MAE  : 0.4242 g CO₂/km
  - RMSE : 2.8000 g CO₂/km
  - R²   : 0.9977

HistGradientBoosting présente une meilleure stabilité entre validation croisée et validatio

### 8.9 Évaluation finale du modèle candidat sur le Test final

### Objectif

À l'issue de la phase de tuning et de validation indépendante, le
`RandomForestRegressor` optimisé a été retenu comme modèle candidat principal
pour la régression des émissions de CO₂ WLTP.

Cette étape consiste à évaluer ce modèle sur le jeu `Test final`, qui n'a pas
été utilisé pendant :

- l'entraînement des configurations candidates ;
- la recherche des hyperparamètres ;
- la validation indépendante des modèles optimisés.

Le jeu `Test final` constitue donc l'évaluation finale de la capacité de
généralisation du modèle retenu.

Les métriques calculées sont :

- MAE ;
- RMSE ;
- R².

Les résultats obtenus seront comparés à ceux observés sur le jeu
`Validation tuning` afin de vérifier que les performances se maintiennent
sur un volume plus important de données totalement indépendant du processus
de sélection.

Aucun nouvel ajustement des hyperparamètres ne sera réalisé à partir des
résultats du Test final.

In [19]:
# ---------------------------------------------------------------------
# 8.9 - Évaluation finale du Random Forest optimisé
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ---------------------------------------------------------------------
# 1. Prédictions sur le Test final
#
# rf_best_model correspond à la meilleure configuration sélectionnée
# lors du tuning puis réentraînée automatiquement sur X_tuning_train
# grâce à refit=True.
#
# Aucun nouvel ajustement d'hyperparamètres n'est effectué ici.
# ---------------------------------------------------------------------

y_rf_test_pred = rf_best_model.predict(
    X_test
)


# ---------------------------------------------------------------------
# 2. Calcul des métriques sur le Test final
# ---------------------------------------------------------------------

rf_test_mae = mean_absolute_error(
    y_test,
    y_rf_test_pred,
)

rf_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_rf_test_pred,
    )
)

rf_test_r2 = r2_score(
    y_test,
    y_rf_test_pred,
)


# ---------------------------------------------------------------------
# 3. Comparaison Validation tuning / Test final
# ---------------------------------------------------------------------

rf_final_evaluation_df = pd.DataFrame(
    [
        {
            "jeu_evaluation": "Validation tuning",
            "observations": len(X_tuning_validation),
            "MAE_g_co2_km": rf_validation_mae,
            "RMSE_g_co2_km": rf_validation_rmse,
            "R2": rf_validation_r2,
        },
        {
            "jeu_evaluation": "Test final",
            "observations": len(X_test),
            "MAE_g_co2_km": rf_test_mae,
            "RMSE_g_co2_km": rf_test_rmse,
            "R2": rf_test_r2,
        },
    ]
)


# ---------------------------------------------------------------------
# 4. Calcul des écarts Validation -> Test
#
# Convention :
#   valeur Test - valeur Validation
#
# MAE / RMSE :
#   > 0 = augmentation de l'erreur
#   < 0 = diminution de l'erreur
#
# R² :
#   > 0 = amélioration du R²
#   < 0 = diminution du R²
# ---------------------------------------------------------------------

rf_final_mae_gap = (
    rf_test_mae
    - rf_validation_mae
)

rf_final_rmse_gap = (
    rf_test_rmse
    - rf_validation_rmse
)

rf_final_r2_gap = (
    rf_test_r2
    - rf_validation_r2
)


# ---------------------------------------------------------------------
# 5. Écart relatif de MAE
# ---------------------------------------------------------------------

rf_final_mae_gap_pct = (
    rf_final_mae_gap
    / rf_validation_mae
    * 100
)


# ---------------------------------------------------------------------
# 6. Version destinée à l'affichage
# ---------------------------------------------------------------------

rf_final_evaluation_display_df = (
    rf_final_evaluation_df.copy()
)

metric_columns = [
    "MAE_g_co2_km",
    "RMSE_g_co2_km",
    "R2",
]

rf_final_evaluation_display_df[
    metric_columns
] = (
    rf_final_evaluation_display_df[
        metric_columns
    ]
    .round(4)
)

display(
    rf_final_evaluation_display_df
)


# ---------------------------------------------------------------------
# 7. Rapport final
# ---------------------------------------------------------------------

print(
    "\nÉVALUATION FINALE DU RANDOM FOREST OPTIMISÉ"
)

print(
    "=" * 70
)

print(
    f"\nMAE Test final  : "
    f"{rf_test_mae:.4f} g CO₂/km"
)

print(
    f"RMSE Test final : "
    f"{rf_test_rmse:.4f} g CO₂/km"
)

print(
    f"R² Test final   : "
    f"{rf_test_r2:.4f}"
)


# ---------------------------------------------------------------------
# 8. Écarts Validation -> Test
# ---------------------------------------------------------------------

print(
    "\nÉCARTS VALIDATION → TEST"
)

print(
    "-" * 70
)

print(
    f"Écart MAE  : "
    f"{rf_final_mae_gap:+.4f} g CO₂/km"
)

print(
    f"Écart RMSE : "
    f"{rf_final_rmse_gap:+.4f} g CO₂/km"
)

print(
    f"Écart R²   : "
    f"{rf_final_r2_gap:+.4f}"
)


# ---------------------------------------------------------------------
# 9. Interprétation dynamique de la MAE
# ---------------------------------------------------------------------

print(
    "\nINTERPRÉTATION"
)

print(
    "-" * 70
)


if rf_final_mae_gap < 0:

    print(
        "✅ La MAE du Test final est inférieure "
        "à celle obtenue sur Validation tuning."
    )

    print(
        f"   L'amélioration est de "
        f"{abs(rf_final_mae_gap):.4f} g CO₂/km "
        f"({abs(rf_final_mae_gap_pct):.2f} %)."
    )

    print(
        "   Aucun signal de dégradation de généralisation "
        "n'est observé sur le Test final."
    )


elif abs(rf_final_mae_gap_pct) <= 5:

    print(
        f"✅ La MAE augmente légèrement de "
        f"{rf_final_mae_gap:.4f} g CO₂/km "
        f"({rf_final_mae_gap_pct:.2f} %) "
        "entre Validation tuning et Test final."
    )

    print(
        "   Les performances restent très proches entre "
        "les deux jeux d'évaluation."
    )

    print(
        "   La généralisation du modèle est donc confirmée."
    )


elif rf_final_mae_gap_pct <= 10:

    print(
        f"⚠️ La MAE augmente de "
        f"{rf_final_mae_gap:.4f} g CO₂/km "
        f"({rf_final_mae_gap_pct:.2f} %) "
        "entre Validation tuning et Test final."
    )

    print(
        "   Cette dégradation reste modérée."
    )


else:

    print(
        f"⚠️ La MAE augmente sensiblement de "
        f"{rf_final_mae_gap:.4f} g CO₂/km "
        f"({rf_final_mae_gap_pct:.2f} %) "
        "entre Validation tuning et Test final."
    )

    print(
        "   Cette dégradation doit être examinée avant "
        "la mise en production."
    )


# ---------------------------------------------------------------------
# 10. Interprétation du R²
# ---------------------------------------------------------------------

if rf_final_r2_gap < 0:

    print(
        f"\nLe R² diminue légèrement de "
        f"{abs(rf_final_r2_gap):.4f} "
        "entre Validation tuning et Test final."
    )

elif rf_final_r2_gap > 0:

    print(
        f"\nLe R² augmente de "
        f"{rf_final_r2_gap:.4f} "
        "entre Validation tuning et Test final."
    )

else:

    print(
        "\nLe R² est identique entre Validation tuning "
        "et Test final."
    )


# ---------------------------------------------------------------------
# 11. Rappel méthodologique
# ---------------------------------------------------------------------

print(
    "\n✅ Le Test final a été utilisé uniquement pour "
    "l'évaluation finale du modèle sélectionné."
)

print(
    "✅ Aucun hyperparamètre n'a été ajusté à partir "
    "des résultats du Test final."
)

,jeu_evaluation,observations,MAE_g_co2_km,RMSE_g_co2_km,R2
0,Validation tuning,100000,0.4242,2.800,0.9977
1,Test final,2151637,0.4378,3.109,0.9971



ÉVALUATION FINALE DU RANDOM FOREST OPTIMISÉ

MAE Test final  : 0.4378 g CO₂/km
RMSE Test final : 3.1090 g CO₂/km
R² Test final   : 0.9971

ÉCARTS VALIDATION → TEST
----------------------------------------------------------------------
Écart MAE  : +0.0135 g CO₂/km
Écart RMSE : +0.3090 g CO₂/km
Écart R²   : -0.0006

INTERPRÉTATION
----------------------------------------------------------------------
✅ La MAE augmente légèrement de 0.0135 g CO₂/km (3.19 %) entre Validation tuning et Test final.
   Les performances restent très proches entre les deux jeux d'évaluation.
   La généralisation du modèle est donc confirmée.

Le R² diminue légèrement de 0.0006 entre Validation tuning et Test final.

✅ Le Test final a été utilisé uniquement pour l'évaluation finale du modèle sélectionné.
✅ Aucun hyperparamètre n'a été ajusté à partir des résultats du Test final.


### 8.10 Préparation du réentraînement FULL du modèle final

### Objectif

À l'issue des étapes d'optimisation, de validation et d'évaluation finale,
le `RandomForestRegressor` optimisé a été retenu comme modèle final de
régression pour la prédiction des émissions de CO₂ WLTP.

Les hyperparamètres utilisés pour le réentraînement sont ceux sélectionnés
lors de la phase d'optimisation précédente.

Ils sont récupérés dynamiquement à partir des résultats du tuning afin
d'éviter toute duplication ou valeur figée dans le notebook.

L'objectif de cette étape est de préparer le réentraînement du modèle final
sur l'intégralité du jeu d'entraînement disponible :

- `X_train` : variables explicatives prétraitées ;
- `y_train` : variable cible `co2_wltp_g_km`.

Aucun nouvel ajustement d'hyperparamètres n'est réalisé à cette étape.
Le modèle est réinstancié à partir de la meilleure configuration obtenue
pendant le tuning.

In [20]:
# ---------------------------------------------------------------------
# 8.10 - Préparation du modèle Random Forest final
# ---------------------------------------------------------------------

from sklearn.ensemble import RandomForestRegressor


# ---------------------------------------------------------------------
# 1. Récupération dynamique des meilleurs hyperparamètres
# ---------------------------------------------------------------------

final_rf_params = rf_best_params.copy()


# ---------------------------------------------------------------------
# 2. Ajout des paramètres de reproductibilité / parallélisation
# ---------------------------------------------------------------------

final_rf_params.update(
    {
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
)


# ---------------------------------------------------------------------
# 3. Instanciation du modèle final
# ---------------------------------------------------------------------

final_random_forest_model = RandomForestRegressor(
    **final_rf_params
)


# ---------------------------------------------------------------------
# 4. Rapport
# ---------------------------------------------------------------------

print("PRÉPARATION DU MODÈLE RANDOM FOREST FINAL")
print("=" * 70)

print(
    f"\nTrain FULL disponible : "
    f"{len(X_train):,} observations"
)

print(
    f"Nombre de variables   : "
    f"{X_train.shape[1]}"
)

print(
    "\nHyperparamètres issus du tuning :"
)

for parameter, value in final_rf_params.items():
    print(
        f"  - {parameter}: {value}"
    )

print(
    "\n✅ Modèle final instancié à partir "
    "des meilleurs hyperparamètres du tuning."
)

print(
    "✅ Le modèle est prêt pour le réentraînement FULL."
)

PRÉPARATION DU MODÈLE RANDOM FOREST FINAL

Train FULL disponible : 8,606,547 observations
Nombre de variables   : 41

Hyperparamètres issus du tuning :
  - n_estimators: 100
  - min_samples_split: 5
  - min_samples_leaf: 1
  - max_features: 1.0
  - max_depth: 25
  - random_state: 42
  - n_jobs: -1

✅ Modèle final instancié à partir des meilleurs hyperparamètres du tuning.
✅ Le modèle est prêt pour le réentraînement FULL.


### 8.11 Industrialisation du réentraînement final

### Objectif

Le modèle final de régression a été sélectionné et ses hyperparamètres optimisés
ont été récupérés dynamiquement à partir du processus de tuning.

L'étape suivante consiste à industrialiser le réentraînement final afin de
pouvoir :

- reproduire l'entraînement en dehors du notebook ;
- exécuter le modèle sur l'intégralité du jeu `Train FULL` ;
- mesurer le temps d'entraînement ;
- sauvegarder le modèle entraîné ;
- enregistrer ses métriques et ses métadonnées ;
- intégrer le modèle au pipeline MLOps.

Le script d'entraînement final devra proposer deux modes :

- **TEST** : validation rapide du fonctionnement sur un sous-échantillon ;
- **FULL** : entraînement sur l'intégralité de `X_train` et `y_train`.

Les hyperparamètres utilisés par le script devront correspondre à la meilleure
configuration sélectionnée lors du tuning.

Le modèle final sera sauvegardé dans le répertoire dédié aux modèles entraînés.

In [22]:
# ---------------------------------------------------------------------
# 8.11 - Préparation des informations du modèle final
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------
# 1. Répertoire de destination du modèle final
# ---------------------------------------------------------------------

trained_models_dir = (
    project_root
    / "models"
    / "trained"
)

trained_models_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 2. Nom du modèle final
# ---------------------------------------------------------------------

FINAL_MODEL_NAME = "random_forest_regressor"


# ---------------------------------------------------------------------
# 3. Hyperparamètres retenus
# ---------------------------------------------------------------------

final_model_params = (
    final_rf_params.copy()
)


# ---------------------------------------------------------------------
# 4. Informations d'entraînement
# ---------------------------------------------------------------------

final_training_config = {
    "model_name": FINAL_MODEL_NAME,
    "target": "co2_wltp_g_km",
    "train_observations": len(X_train),
    "train_features": X_train.shape[1],
    "test_observations": len(X_test),
    "test_features": X_test.shape[1],
    "model_directory": str(trained_models_dir),
    "hyperparameters": final_model_params,
}


# ---------------------------------------------------------------------
# 5. Tableau synthétique
#
# Le chemin complet du répertoire n'est volontairement pas placé
# dans le tableau afin d'éviter sa troncature à l'affichage.
# ---------------------------------------------------------------------

final_training_report_df = pd.DataFrame(
    [
        {
            "element": "Modèle final",
            "valeur": FINAL_MODEL_NAME,
        },
        {
            "element": "Cible",
            "valeur": "co2_wltp_g_km",
        },
        {
            "element": "Train FULL",
            "valeur": f"{len(X_train):,} observations",
        },
        {
            "element": "Test final",
            "valeur": f"{len(X_test):,} observations",
        },
        {
            "element": "Nombre de variables",
            "valeur": X_train.shape[1],
        },
    ]
)

display(
    final_training_report_df
)


# ---------------------------------------------------------------------
# 6. Affichage explicite du répertoire de destination
# ---------------------------------------------------------------------

print("\nRÉPERTOIRE DE DESTINATION DU MODÈLE")
print("=" * 70)

print(
    trained_models_dir
)


# ---------------------------------------------------------------------
# 7. Hyperparamètres à industrialiser
# ---------------------------------------------------------------------

print("\nHYPERPARAMÈTRES À INDUSTRIALISER")
print("=" * 70)

for parameter, value in final_model_params.items():
    print(
        f"  - {parameter}: {value}"
    )


# ---------------------------------------------------------------------
# 8. Vérification de la cohérence
# ---------------------------------------------------------------------

if final_training_config["train_features"] != X_test.shape[1]:
    raise ValueError(
        "Le nombre de variables Train et Test est différent."
    )

if not trained_models_dir.is_dir():
    raise FileNotFoundError(
        "Le répertoire models/trained n'a pas pu être créé."
    )


# ---------------------------------------------------------------------
# 9. Rapport final
# ---------------------------------------------------------------------

print(
    "\n✅ Configuration du modèle final préparée."
)

print(
    "✅ Structure Train / Test cohérente."
)

print(
    "✅ Répertoire de destination du modèle disponible."
)

print(
    "✅ Aucun entraînement FULL n'a encore été lancé."
)

,element,valeur
0,Modèle final,random_forest_regressor
1,Cible,co2_wltp_g_km
2,Train FULL,"8,606,547 observations"
3,Test final,"2,151,637 observations"
4,Nombre de variables,41



RÉPERTOIRE DE DESTINATION DU MODÈLE
/home/jmbandong/projects/ml-projects/vehicle-emissions-prediction-mlops/models/trained

HYPERPARAMÈTRES À INDUSTRIALISER
  - n_estimators: 100
  - min_samples_split: 5
  - min_samples_leaf: 1
  - max_features: 1.0
  - max_depth: 25
  - random_state: 42
  - n_jobs: -1

✅ Configuration du modèle final préparée.
✅ Structure Train / Test cohérente.
✅ Répertoire de destination du modèle disponible.
✅ Aucun entraînement FULL n'a encore été lancé.


### 8.12 Sauvegarde des résultats et métadonnées de l'expérimentation

### Objectif

Les différentes étapes du notebook ont produit plusieurs résultats nécessaires
à la traçabilité de l'expérimentation de régression.

Avant l'industrialisation du modèle final, les principaux résultats sont
sauvegardés afin de conserver :

- la comparaison initiale des modèles ;
- les résultats du tuning du Random Forest ;
- les résultats du tuning du HistGradientBoosting ;
- la comparaison des modèles optimisés ;
- les hyperparamètres sélectionnés ;
- les métriques finales obtenues sur le jeu de test.

Les résultats tabulaires sont sauvegardés dans :

`reports/tables/`

Les métadonnées de l'expérimentation sont sauvegardées dans :

`models/metadata/`

Cette étape permet de séparer :

- les résultats d'expérimentation ;
- les métadonnées du modèle sélectionné ;
- le futur artefact du modèle FULL, qui sera produit lors de l'entraînement
  industrialisé.

In [23]:
# ---------------------------------------------------------------------
# 8.12 - Sauvegarde des résultats et métadonnées du notebook 04
# ---------------------------------------------------------------------

import json
from pathlib import Path

import pandas as pd


# ---------------------------------------------------------------------
# 1. Répertoires de destination
# ---------------------------------------------------------------------

reports_tables_dir = (
    project_root
    / "reports"
    / "tables"
)

models_metadata_dir = (
    project_root
    / "models"
    / "metadata"
)

reports_tables_dir.mkdir(
    parents=True,
    exist_ok=True,
)

models_metadata_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 2. Fichiers de résultats tabulaires
# ---------------------------------------------------------------------

regression_comparison_path = (
    reports_tables_dir
    / "regression_model_comparison.csv"
)

rf_tuning_results_path = (
    reports_tables_dir
    / "random_forest_tuning_results.csv"
)

hgb_tuning_results_path = (
    reports_tables_dir
    / "hist_gradient_boosting_tuning_results.csv"
)

optimized_models_comparison_path = (
    reports_tables_dir
    / "optimized_models_comparison.csv"
)

rf_final_evaluation_path = (
    reports_tables_dir
    / "random_forest_final_evaluation.csv"
)


# ---------------------------------------------------------------------
# 3. Sauvegarde des tableaux
# ---------------------------------------------------------------------

regression_comparison_df.to_csv(
    regression_comparison_path,
    index=False,
)

rf_tuning_results_df.to_csv(
    rf_tuning_results_path,
    index=False,
)

hgb_tuning_results_df.to_csv(
    hgb_tuning_results_path,
    index=False,
)

optimized_models_comparison_df.to_csv(
    optimized_models_comparison_path,
    index=False,
)

rf_final_evaluation_df.to_csv(
    rf_final_evaluation_path,
    index=False,
)


# ---------------------------------------------------------------------
# 4. Métadonnées de l'expérimentation
# ---------------------------------------------------------------------

regression_metadata = {
    "model_selected": "RandomForestRegressor",

    "target": "co2_wltp_g_km",

    "train_full": {
        "observations": int(len(X_train)),
        "features": int(X_train.shape[1]),
    },

    "test_final": {
        "observations": int(len(X_test)),
        "features": int(X_test.shape[1]),
    },

    "tuning": {
        "train_observations": int(len(X_tuning_train)),
        "validation_observations": int(
            len(X_tuning_validation)
        ),
        "cv_folds": int(CV_FOLDS),
        "scoring": SCORING,
    },

    "best_random_forest_params": {
        key: (
            value.item()
            if hasattr(value, "item")
            else value
        )
        for key, value in rf_best_params.items()
    },

    "final_test_metrics": {
        "mae": float(rf_test_mae),
        "rmse": float(rf_test_rmse),
        "r2": float(rf_test_r2),
    },
}


# ---------------------------------------------------------------------
# 5. Sauvegarde des métadonnées JSON
# ---------------------------------------------------------------------

regression_metadata_path = (
    models_metadata_dir
    / "random_forest_regression_metadata.json"
)

with open(
    regression_metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        regression_metadata,
        file,
        indent=4,
        ensure_ascii=False,
    )


# ---------------------------------------------------------------------
# 6. Rapport de sauvegarde
# ---------------------------------------------------------------------

saved_results_df = pd.DataFrame(
    [
        {
            "type": "Comparaison modèles",
            "fichier": regression_comparison_path.name,
            "repertoire": "reports/tables",
        },
        {
            "type": "Tuning Random Forest",
            "fichier": rf_tuning_results_path.name,
            "repertoire": "reports/tables",
        },
        {
            "type": "Tuning HistGradientBoosting",
            "fichier": hgb_tuning_results_path.name,
            "repertoire": "reports/tables",
        },
        {
            "type": "Comparaison modèles optimisés",
            "fichier": optimized_models_comparison_path.name,
            "repertoire": "reports/tables",
        },
        {
            "type": "Évaluation finale Random Forest",
            "fichier": rf_final_evaluation_path.name,
            "repertoire": "reports/tables",
        },
        {
            "type": "Métadonnées régression",
            "fichier": regression_metadata_path.name,
            "repertoire": "models/metadata",
        },
    ]
)

display(
    saved_results_df
)


print(
    "\n✅ Résultats expérimentaux sauvegardés."
)

print(
    "✅ Métadonnées du modèle sélectionné sauvegardées."
)

print(
    "✅ Aucun modèle FULL n'a été sauvegardé à cette étape."
)

,type,fichier,repertoire
0,Comparaison modèles,regression_model_comparison.csv,reports/tables
1,Tuning Random Forest,random_forest_tuning_results.csv,reports/tables
2,Tuning HistGradientBoosting,hist_gradient_boosting_tuning_results.csv,reports/tables
3,Comparaison modèles optimisés,optimized_models_comparison.csv,reports/tables
4,Évaluation finale Random Forest,random_forest_final_evaluation.csv,reports/tables
5,Métadonnées régression,random_forest_regression_metadata.json,models/metadata



✅ Résultats expérimentaux sauvegardés.
✅ Métadonnées du modèle sélectionné sauvegardées.
✅ Aucun modèle FULL n'a été sauvegardé à cette étape.


### 8.13 Validation des résultats sauvegardés

### Objectif

Cette étape vérifie que les fichiers produits à l'étape précédente existent
bien, peuvent être relus et contiennent les informations attendues.

La validation porte sur :

- les tableaux CSV de comparaison et de tuning ;
- le fichier JSON de métadonnées ;
- la présence des métriques finales ;
- la présence des hyperparamètres sélectionnés.

Cette étape permet de s'assurer que les résultats du notebook sont correctement
persistés avant l'industrialisation de l'entraînement final.

In [ ]:
# ---------------------------------------------------------------------
# 6. Rapport de validation
# ---------------------------------------------------------------------

validation_report = []

for name, result in checks.items():

    validation_report.append(
        {
            "controle": name,
            "statut": (
                "Sauvegardé et validé"
                if result
                else "Erreur de validation"
            ),
        }
    )


validation_report_df = pd.DataFrame(
    validation_report
)

display(
    validation_report_df
)

,controle,statut
0,Comparaison modèles rechargeable,Sauvegardé et validé
1,Tuning Random Forest rechargeable,Sauvegardé et validé
2,Tuning HistGradientBoosting rechargeable,Sauvegardé et validé
3,Comparaison modèles optimisés rechargeable,Sauvegardé et validé
4,Évaluation finale rechargeable,Sauvegardé et validé
5,Modèle sélectionné présent,Sauvegardé et validé
6,Hyperparamètres présents,Sauvegardé et validé
7,Métriques finales présentes,Sauvegardé et validé


: 